# Adaptive Multimodal Sepsis Early-Warning System — v4 (paper-safety experiments)
**Reliability-aware clinical decision support for early sepsis risk estimation** · AINN project

### What v4 adds (every model from v3 is kept; v4 only adds experiments that prove or disprove each claim)
| Section | Experiment | The reviewer question it answers |
|---|---|---|
| 14 | **Shortcut ablation**: retrain without time-in-ICU (ICULOS, HospAdmTime) and without "when was it measured" features | "Is the model just learning how long patients stayed, or which tests doctors ordered?" |
| 15 | **5 seeds per neural model + paired bootstrap tests** (internal, stress, external hospital B) | "Is the hybrid really better than LightGBM, or is it noise?" |
| 16 | **Trigger efficiency curve** for evidence-seeking (REVIEW rule vs risk-only vs random) | "Does the reliability trigger request labs more efficiently than a plain risk threshold?" |
| 17 | **Selective-prediction diagnostics**: where do deferred hours go? | "Does deferral just hide the sepsis cases?" |
| 18 | **Conformal REVIEW**: class-conditional conformal thresholds with a target miss rate | "Your REVIEW cut-offs are hand-picked." |
| 19 | **Irrelevant-evidence control**: add a pure-noise stream and see what the gate does | "Does the reliability gate actually down-weight useless evidence?" |

All test numbers come from untouched real patients. Augmentation (modality dropout) is used in training only.

**Run on Kaggle:** GPU T4 · Internet ON · input `salikhussaini49/prediction-of-sepsis` → **Save Version → Save & Run All (Commit)**.
Expected time ≈ 3 h (X-ray branch is off by default because its results are already saved from v3; set `RUN_IMAGING = True` in cell 1 to include it).
Outputs → `/kaggle/working/results/`; `SLIDE_SUMMARY.md` now includes every v4 table.

---
#### Earlier changes (v2 → v3)

### What changed from v2 (based on the first real Kaggle run)
| v2 finding | v3 change |
|---|---|
| LightGBM (0.839) beat every neural model (best 0.817) | **Hybrid risk engine** = LightGBM seed-ensemble + reliability-gated GRU-D, stacked on validation. Our contribution is the *decision layer*, so it sits on top of the strongest risk engine |
| Neural training loss barely moved, validation noisy | Random length-buckets (v2 sorted batches by length), OneCycle LR, wider model, **GRU-D input decay** for missing values |
| MC-dropout σ alone just re-ranks by risk | Uncertainty = **ensemble × MC-dropout** predictive spread; evidence ranking compared against a "defer early ICU hours" baseline |
| REVIEW lowered utility (0.403 → 0.335) | REVIEW redefined as an **evidence-seeking action** ("obtain labs / reassess") and tested counterfactually: under a lab outage, requested labs are revealed |
| Calibration plot had 2 points; ECE ≈ 0.001 was meaningless | Equal-mass bins + adaptive ECE |
| Utility-optimal threshold alerts on 19% of hours | Adds a **low-burden operating point** (≤5% alert hours) |
| No clinical comparator, no external validation | **SIRS / qSOFA-lite** baselines + **train hospital A → test hospital B** |
| Cell 13 crashed (`cudnn RNN backward … training mode`) | Explanations now use exact **TreeSHAP** from LightGBM (no RNN backward) |

## 1 · Setup

In [ ]:
import os, sys, json, time, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')

INPUT_ROOT = os.environ.get('SEPSIS_INPUT_ROOT', '/kaggle/input')
WORK_DIR   = os.environ.get('SEPSIS_WORK_DIR', '/kaggle/working')
QUICK      = os.environ.get('SEPSIS_QUICK', '0') == '1'      # keep 0 on Kaggle
OUT_DIR = os.path.join(WORK_DIR, 'results'); FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFG = dict(
    max_epochs   = 2 if QUICK else 30,
    patience     = 6,
    batch_size   = 64,
    lr           = 5e-4,       # peak LR of the OneCycle schedule
    weight_decay = 1e-4,
    hidden       = 128,
    dropout      = 0.3,
    mc_passes    = 5 if QUICK else 30,
    n_boot       = 20 if QUICK else 100,
    aug_p_labs = 0.25, aug_p_sparse_vitals = 0.15, aug_p_random = 0.20, aug_random_drop = 0.30,
    lgb_seeds    = 2 if QUICK else 5,
    lgb_rounds   = 80 if QUICK else 3000,
    max_alert_rate = 0.05,     # low-burden operating point: at most 5% of hours in ALERT
    # ---- v4 paper-safety experiments ----
    n_seeds        = 2 if QUICK else 5,          # neural runs per model (seed 0 = main run)
    n_pair_boot    = 20 if QUICK else 200,       # paired bootstrap resamples
    trigger_mults  = [0.5, 1, 2] if QUICK else [0.25, 0.5, 1, 2, 4],
    random_fracs   = [0.25, 0.5] if QUICK else [0.1, 0.25, 0.5, 0.75],
    conformal_alphas = [0.1, 0.2, 0.3] if QUICK else [0.05, 0.1, 0.2, 0.3, 0.4],
)
RUN_CROSS_HOSPITAL = True
RUN_IMAGING        = False   # X-ray results already saved from v3; set True to re-run (~50 min)
RUN_ABLATION = RUN_SEEDS = RUN_TRIGGER_CURVE = RUN_SELECTIVE_DIAG = RUN_CONFORMAL = RUN_IRRELEVANT = True
IMG_PRETRAINED     = os.environ.get('SEPSIS_IMG_PRETRAINED', '1') == '1'

VITALS = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2']
LABS   = ['BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2','AST','BUN','Alkalinephos','Calcium',
          'Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate','Magnesium','Phosphate',
          'Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
DEMO   = ['Age','Gender','Unit1','Unit2','HospAdmTime','ICULOS']
DYN    = VITALS + LABS
NV, NL = len(VITALS), len(LABS)
LABEL  = 'SepsisLabel'

MODEL_NAMES = {'sirs':   'SIRS ≥2 (clinical rule)',
               'qsofa':  'qSOFA-lite ≥2 (clinical rule)',
               'lgbm':   'LightGBM ensemble',
               'gru':    'GRU early fusion',
               'gru_aug':'GRU early fusion + modality dropout',
               'rgn':    'Reliability-gated GRU-D',
               'hybrid': 'Hybrid LightGBM + RG-GRU-D (proposed)'}
RULES = ['sirs', 'qsofa']
COLORS = {'sirs': '#b07aa1', 'qsofa': '#9c755f', 'lgbm': '#8c8c8c', 'gru': '#4e79a7',
          'gru_aug': '#76b7b2', 'rgn': '#f28e2b', 'hybrid': '#e15759'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 10})
def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=160, bbox_inches='tight')
def show_table(df):
    display(df) if 'display' in globals() else print(df.to_string())

print(f'device={device} | torch {torch.__version__} | QUICK={QUICK} | CPUs={os.cpu_count()}')
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

## 2 · Load PhysioNet 2019 (cached after the first run)

In [ ]:
def find_files(root, exts, skip_images=True):
    out = []
    for dp, dns, fns in os.walk(root, followlinks=True):
        if skip_images:
            dns[:] = [d for d in dns if not d.lower().startswith('images')]
        out += [os.path.join(dp, f) for f in fns if f.lower().endswith(exts)]
    return sorted(out)

def read_psv(path):
    df = pd.read_csv(path, sep='|')
    pid = os.path.splitext(os.path.basename(path))[0]
    df['patient_id'] = pid
    df['hospital'] = 'B' if pid.startswith('p1') else 'A'   # p0xxxxx = hospital A, p1xxxxx = hospital B
    df['t'] = np.arange(len(df))
    return df

CACHE = os.path.join(WORK_DIR, 'sepsis_raw_cache.pkl')
t0 = time.time()
if os.path.exists(CACHE):
    raw = pd.read_pickle(CACHE); print('Loaded cached table')
else:
    psv = find_files(INPUT_ROOT, ('.psv',))
    uniq = {}
    for p in psv: uniq.setdefault(os.path.basename(p), p)
    psv = sorted(uniq.values())
    if len(psv) > 0:
        from joblib import Parallel, delayed
        print(f'Reading {len(psv):,} .psv files in parallel ...')
        raw = pd.concat(Parallel(n_jobs=-1, batch_size=256)(delayed(read_psv)(p) for p in psv), ignore_index=True)
    else:
        csvs = [p for p in find_files(INPUT_ROOT, ('.csv',)) if LABEL in pd.read_csv(p, nrows=0).columns]
        assert csvs, 'No sepsis data found. Attach the "Prediction of Sepsis" dataset via + Add Input.'
        raw = pd.read_csv(csvs[0])
        raw = raw.drop(columns=[c for c in raw.columns if c.startswith('Unnamed')])
        num = pd.to_numeric(raw['Patient_ID'], errors='coerce')
        raw['hospital'] = np.where(num >= 100000, 'B', 'A')
        raw['patient_id'] = raw['Patient_ID'].astype(str)
        order = 'Hour' if 'Hour' in raw.columns else 'ICULOS'
        raw = raw.sort_values(['patient_id', order], kind='stable').reset_index(drop=True)
        raw['t'] = raw.groupby('patient_id').cumcount()
    raw = raw.sort_values(['patient_id', 't'], kind='stable').reset_index(drop=True)
    raw[DYN + DEMO] = raw[DYN + DEMO].astype(np.float32)
    raw[LABEL] = raw[LABEL].astype(np.int8)
    raw = raw[['patient_id', 'hospital', 't'] + DYN + DEMO + [LABEL]]
    raw.to_pickle(CACHE)
print(f'{len(raw):,} patient-hours loaded in {time.time()-t0:.1f}s')

## 3 · Dataset audit

In [ ]:
pid_arr = raw['patient_id'].values
starts = np.r_[0, np.flatnonzero(pid_arr[1:] != pid_arr[:-1]) + 1]
ends   = np.r_[starts[1:], len(raw)]
y_all  = raw[LABEL].values.astype(np.int8)
t_all  = raw['t'].values.astype(np.int32)
pat = pd.DataFrame({'patient_id': pid_arr[starts], 'start': starts, 'end': ends,
                    'length': ends - starts, 'septic': np.maximum.reduceat(y_all, starts),
                    'hospital': raw['hospital'].values[starts]})
N_ROWS, N_PAT = len(raw), len(pat)
miss = raw[DYN].isna().mean().sort_values()
summary = {
    'patients': int(N_PAT), 'patient_hours': int(N_ROWS),
    'septic_patients': int(pat.septic.sum()), 'septic_patient_pct': round(100 * pat.septic.mean(), 2),
    'positive_hours_pct': round(100 * y_all.mean(), 2),
    'hospital_A_patients': int((pat.hospital == 'A').sum()), 'hospital_B_patients': int((pat.hospital == 'B').sum()),
    'median_stay_h': float(pat.length.median()), 'max_stay_h': int(pat.length.max()),
    'vitals_missing_pct': round(100 * raw[VITALS].isna().values.mean(), 1),
    'labs_missing_pct': round(100 * raw[LABS].isna().values.mean(), 1),
}
json.dump(summary, open(os.path.join(OUT_DIR, 'dataset_summary.json'), 'w'), indent=2)
for k, v in summary.items(): print(f'{k:>22}: {v}')
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(len(miss)), 100 * miss.values, color=['#4e79a7' if v in VITALS else '#f28e2b' for v in miss.index])
ax.set_xticks(range(len(miss))); ax.set_xticklabels(miss.index, rotation=75, fontsize=8)
ax.set_ylabel('% hours missing'); ax.set_title('Missingness per variable (blue = vitals, orange = labs)')
savefig('01_missingness.png'); plt.show()

## 4 · Splits + preprocessing
* Internal split: patient-level 70/15/15, stratified by septic × hospital (same as v2, same seed).
* External split (section 13): train on **hospital A** only, test on **all hospital B patients**.

In [ ]:
strat = pat.septic.astype(str) + pat.hospital
tr, tmp = train_test_split(np.arange(N_PAT), test_size=0.30, random_state=SEED, stratify=strat)
va, te  = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=strat.iloc[tmp])
SPLITS = {'train': np.sort(tr), 'val': np.sort(va), 'test': np.sort(te)}
isA = pat.hospital.values == 'A'
SPLITS.update({'A_train': SPLITS['train'][isA[SPLITS['train']]], 'A_val': SPLITS['val'][isA[SPLITS['val']]],
               'A_test': SPLITS['test'][isA[SPLITS['test']]], 'B_all': np.flatnonzero(~isA)})
json.dump({k: pat.patient_id.values[v].tolist() for k, v in SPLITS.items() if k in ('train', 'val', 'test')},
          open(os.path.join(OUT_DIR, 'split_patient_ids.json'), 'w'))

class SplitInfo:
    def __init__(self, pidx):
        self.pidx = pidx
        self.lengths = pat.length.values[pidx]
        self.off = np.r_[0, np.cumsum(self.lengths)]
        self.rows = np.concatenate([np.arange(pat.start.values[p], pat.end.values[p]) for p in pidx]) if len(pidx) else np.zeros(0, int)
        self.y = y_all[self.rows]
        self.septic = pat.septic.values[pidx]
        self.pos = {p: i for i, p in enumerate(pidx)}
SI = {k: SplitInfo(v) for k, v in SPLITS.items()}
for k in ['train', 'val', 'test']:
    s = SI[k]; print(f'{k:>5}: {len(s.pidx):6,} patients | {len(s.rows):9,} hours | septic {s.septic.mean()*100:5.2f}% | positive hours {s.y.mean()*100:4.2f}%')

X_raw = raw[DYN].values.astype(np.float32)
D_raw = raw[DEMO].values.astype(np.float32)
tr_rows = SI['train'].rows
def _stats(v):
    v = v[~np.isnan(v)]
    if not len(v): return 0., 1., 0., 1.
    lo, hi = np.percentile(v, [0.5, 99.5]); vc = np.clip(v, lo, hi)
    return float(lo), float(hi), float(vc.mean()), float(vc.std() + 1e-6)
st = np.array([_stats(X_raw[tr_rows, j]) for j in range(len(DYN))])
NORM = {'lo': st[:, 0], 'hi': st[:, 1], 'mu': st[:, 2], 'sd': st[:, 3]}
for k, j in [('age', 0), ('hadm', 4), ('icu', 5)]:
    lo, hi, mu, sd = _stats(D_raw[tr_rows, j]); NORM.update({k+'_lo': lo, k+'_hi': hi, k+'_mu': mu, k+'_sd': sd})

def to_model_inputs(Xr, Dm, norm=NORM):
    """Raw values (NaN = not measured) -> z-scored values Z, mask M, context C. Shared by training and the live monitor."""
    M = ~np.isnan(Xr)
    Z = np.where(M, (np.clip(Xr, norm['lo'], norm['hi']) - norm['mu']) / norm['sd'], 0.).astype(np.float32)
    age, gender, u1, u2, hadm, icu = [Dm[:, j] for j in range(6)]
    zc = lambda v, k: np.nan_to_num((np.clip(v, norm[k+'_lo'], norm[k+'_hi']) - norm[k+'_mu']) / norm[k+'_sd'])
    C = np.stack([zc(age, 'age'), np.nan_to_num(gender), np.nan_to_num(u1), np.nan_to_num(u2),
                  (np.isnan(u1) & np.isnan(u2)).astype(np.float32), zc(hadm, 'hadm'), zc(icu, 'icu')], 1).astype(np.float32)
    return Z, M, C
Z_all, M_all, C_all = to_model_inputs(X_raw, D_raw)
print('Z', Z_all.shape, '| M', M_all.shape, '| C', C_all.shape)

## 5 · Evaluation toolkit
Official PhysioNet utility (vectorised), AUROC/AUPRC with patient-bootstrap 95% CIs, patient-level AUROC,
Brier, **adaptive ECE** (equal-mass bins), detection rate, median lead time, false-alarm burden, and two operating
points: **utility-optimal** and **low-burden** (≤5% of hours in ALERT, chosen on validation).

In [ ]:
U_POS = np.zeros(N_ROWS, np.float32); U_NEG = np.zeros(N_ROWS, np.float32); BEST = np.zeros(N_ROWS, bool)
for s, e, sep in zip(pat.start.values, pat.end.values, pat.septic.values):
    if not sep:
        U_POS[s:e] = -0.05; continue
    n = e - s; ts = int(np.argmax(y_all[s:e])) + 6
    d = np.arange(n) - ts
    U_POS[s:e] = np.where(d <= 3, np.where(d <= -6, np.maximum(d / 6 + 2, -0.05), -d / 9 + 1 / 3), 0.)
    U_NEG[s:e] = np.where(d <= 3, np.where(d <= -6, 0., -2 / 9 * d - 4 / 3), 0.)
    BEST[s + max(0, ts - 12): s + min(ts + 4, n)] = True

def utility(S, pred):
    r = S.rows
    obs = np.where(pred, U_POS[r], U_NEG[r]).sum(); best = np.where(BEST[r], U_POS[r], U_NEG[r]).sum(); ina = U_NEG[r].sum()
    return float((obs - ina) / (best - ina))

def threshold_grid(p):
    return np.unique(np.quantile(p, np.linspace(0.50, 0.999, 250)))
def best_threshold(S, p):
    g = threshold_grid(p); return float(g[int(np.argmax([utility(S, p >= x) for x in g]))])
def low_burden_threshold(S, p, max_rate=CFG['max_alert_rate']):
    g = [x for x in threshold_grid(p) if (p >= x).mean() <= max_rate]
    return float(g[int(np.argmax([utility(S, p >= x) for x in g]))]) if g else float(np.quantile(p, 1 - max_rate))

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -30, 30)))
def fit_platt(logit, y):
    lr = LogisticRegression(C=1e4, max_iter=1000).fit(logit.reshape(-1, 1), y)
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])

def ece(p, y, bins=15):
    """Adaptive ECE: equal-mass bins (uniform bins put ~all sepsis predictions in one bin)."""
    order = np.argsort(p); chunks = np.array_split(order, bins)
    return float(sum(abs(p[c].mean() - y[c].mean()) * len(c) / len(p) for c in chunks if len(c)))

def boot_ci(S, p, B=None):
    B = B or CFG['n_boot']; rng = np.random.default_rng(0)
    ranges = [np.arange(S.off[i], S.off[i + 1]) for i in range(len(S.pidx))]
    au, ap = [], []
    for _ in range(B):
        r = np.concatenate([ranges[i] for i in rng.integers(0, len(ranges), len(ranges))]); yy = S.y[r]
        if yy.min() == yy.max(): continue
        au.append(roc_auc_score(yy, p[r])); ap.append(average_precision_score(yy, p[r]))
    return np.percentile(au, [2.5, 97.5]), np.percentile(ap, [2.5, 97.5])

def alert_stats(S, alert):
    det = nsep = nneg = fa_pat = fa_ep = 0; leads = []; neg_days = 0.
    for i in range(len(S.pidx)):
        a = alert[S.off[i]:S.off[i + 1]]; y = S.y[S.off[i]:S.off[i + 1]]
        if S.septic[i]:
            nsep += 1; ts = int(np.argmax(y)) + 6
            w = np.flatnonzero(a[:min(len(a), ts + 4)])
            if len(w): det += 1; leads.append(ts - w[0])
        else:
            nneg += 1; fa_pat += int(a.any())
            fa_ep += int(np.sum(np.diff(np.r_[0, a.astype(np.int8)]) == 1)); neg_days += len(a) / 24
    return {'detection_rate': det / max(nsep, 1),
            'median_lead_h': float(np.median(leads)) if leads else float('nan'),
            'false_alarm_patient_rate': fa_pat / max(nneg, 1),
            'false_alerts_per_patient_day': fa_ep / max(neg_days, 1e-9),
            'alert_precision_hour': float(S.y[alert].mean()) if alert.any() else float('nan'),
            'alert_hour_rate': float(alert.mean())}

def evaluate(S, p, thr, ci=True, prob=True):
    r = {'AUROC': roc_auc_score(S.y, p), 'AUPRC': average_precision_score(S.y, p)}
    if ci:
        (a0, a1), (p0, p1) = boot_ci(S, p); r.update({'AUROC_lo': a0, 'AUROC_hi': a1, 'AUPRC_lo': p0, 'AUPRC_hi': p1})
    r.update({'patient_AUROC': roc_auc_score(S.septic, np.maximum.reduceat(p, S.off[:-1])),
              'utility': utility(S, p >= thr), 'threshold': thr,
              'Brier': float(np.mean((p - S.y) ** 2)) if prob else np.nan, 'ECE': ece(p, S.y) if prob else np.nan})
    r.update(alert_stats(S, p >= thr))
    return r
print('perfect-prediction utility check:', round(utility(SI['val'], BEST[SI['val'].rows]), 3), '(must be 1.0)')

## 6 · Tree models + clinical rules
**Causal features** (only data up to hour *t*): last value, hours since measured, 6-h vitals window stats,
running worst vitals, 24-h lab trends, shock index, **SIRS** count, **qSOFA-lite** (resp ≥22 + SBP ≤100; the GCS
component is not in the dataset). SIRS and qSOFA-lite are also reported as the **clinical baselines** hospitals use today.
LightGBM is trained as a **5-seed ensemble** (different bagging/feature subsamples) — the spread between members
is part of our uncertainty estimate.

In [ ]:
PRED, AB, THR, THR_LB = {}, {}, {}, {}
pidc_all = pd.factorize(raw['patient_id'])[0]
KEY_LABS = ['Lactate', 'WBC', 'Creatinine', 'Platelets', 'Bilirubin_total', 'BUN']

def lgb_feats_from(Xr, Mk, g, t, Dm):
    """Xr raw values, Mk observation mask, g patient codes, t hour index, Dm demographics -> feature table."""
    Xr = Xr.copy(); Xr[~Mk] = np.nan
    D = pd.DataFrame(Xr, columns=DYN)
    ff = D.groupby(g).ffill()
    obs_t = pd.DataFrame(np.where(Mk, t[:, None].astype(np.float32), np.nan), columns=DYN).groupby(g).ffill().values
    delta = np.where(np.isnan(obs_t), 999., t[:, None] - obs_t)
    fv = ff[VITALS]; gv = fv.groupby(g)
    sh = np.stack([fv.values] + [gv.shift(k).values for k in range(1, 6)], 0)
    with np.errstate(all='ignore'), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        win = [np.nanmean(sh, 0), np.nanstd(sh, 0), np.nanmax(sh, 0) - np.nanmin(sh, 0), fv.values - sh[-1]]
    cmax = fv[['HR', 'Temp', 'Resp']].groupby(g).cummax().values
    cmin = fv[['MAP', 'SBP', 'O2Sat']].groupby(g).cummin().values
    kl = ff[KEY_LABS]; ltr = kl.values - kl.groupby(g).shift(24).values
    HR, SBP, Tp, RR, PaCO2, WBC = (ff[c].values for c in ['HR', 'SBP', 'Temp', 'Resp', 'PaCO2', 'WBC'])
    with np.errstate(all='ignore'):
        sirs = ((Tp > 38) | (Tp < 36)).astype(np.int8) + (HR > 90) + ((RR > 20) | (PaCO2 < 32)) + ((WBC > 12) | (WBC < 4))
        qsofa = (RR >= 22).astype(np.int8) + (SBP <= 100)
        shock = HR / SBP
    labs_seen = pd.DataFrame(Mk[:, NV:].astype(np.int8)).groupby(g).cummax().sum(1).values
    cols = ([ff.values, delta] + win + [cmax, cmin, ltr, np.c_[shock, sirs, qsofa, labs_seen], Dm])
    names = (DYN + [f'{v}_dt' for v in DYN] + [f'{v}_{s}6h' for s in ['mean', 'std', 'range', 'trend'] for v in VITALS]
             + ['HR_max', 'Temp_max', 'Resp_max', 'MAP_min', 'SBP_min', 'O2Sat_min'] + [f'{v}_trend24h' for v in KEY_LABS]
             + ['shock_index', 'sirs', 'qsofa', 'labs_seen'] + DEMO)
    return pd.DataFrame(np.hstack(cols).astype(np.float32), columns=names)

def lgb_features(rows, M):
    return lgb_feats_from(X_raw[rows], M[rows], pidc_all[rows], t_all[rows], D_raw[rows])

import lightgbm as lgb
def train_lgb(Ftr, ytr, Fva, yva, seeds):
    ratio = (1 - ytr.mean()) / ytr.mean()
    dtr = lgb.Dataset(Ftr, ytr, free_raw_data=False); dva = lgb.Dataset(Fva, yva, reference=dtr, free_raw_data=False)
    models = []
    for s in seeds:
        params = dict(objective='binary', learning_rate=0.05, num_leaves=31, min_data_in_leaf=300,
                      feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1, lambda_l2=5.0,
                      scale_pos_weight=float(np.sqrt(ratio)), metric=['auc', 'average_precision'],
                      verbose=-1, seed=int(s), num_threads=os.cpu_count())
        m = lgb.train(params, dtr, num_boost_round=CFG['lgb_rounds'], valid_sets=[dva],
                      callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)])
        models.append(m); print(f'  seed {s}: {m.best_iteration} trees | val AUROC {m.best_score["valid_0"]["auc"]:.4f}')
    return models
def lgb_members(models, Fx):
    return np.stack([m.predict(Fx, raw_score=True, num_iteration=m.best_iteration) for m in models]).astype(np.float32)

t0 = time.time()
FEAT = {k: lgb_features(SI[k].rows, M_all) for k in ['train', 'val', 'test']}
print(f'features: {FEAT["train"].shape[1]} columns, built in {time.time()-t0:.0f}s')
LGB_MODELS = train_lgb(FEAT['train'], SI['train'].y, FEAT['val'], SI['val'].y, [SEED + i for i in range(CFG['lgb_seeds'])])
for k in ['val', 'test']:
    mem = lgb_members(LGB_MODELS, FEAT[k])
    PRED.setdefault('lgbm', {})[k] = {'full': {'logit': mem.mean(0), 'members': mem}}
    for r in RULES: PRED.setdefault(r, {})[k] = {'full': {'score': FEAT[k][r].fillna(0).values.astype(np.float32)}}
for i, m in enumerate(LGB_MODELS): m.save_model(os.path.join(OUT_DIR, f'lightgbm_seed{i}.txt'))
imp = pd.Series(np.mean([m.feature_importance('gain') for m in LGB_MODELS], 0), index=FEAT['train'].columns).sort_values(ascending=False)
imp.to_csv(os.path.join(OUT_DIR, 'lightgbm_feature_importance.csv'))
print(f'LightGBM ensemble done in {time.time()-t0:.0f}s | top features: ' + ', '.join(imp.index[:10]))
del FEAT['train']

## 7 · Neural data pipeline
Causal forward-fill, "measured now" mask, time since last measurement, per-modality reliability signals
(`obs_now`, `recency`, `coverage`) and the **evidence score** `E(t) = ½·recency(vitals) + ½·coverage(labs)`.
**Batching fix:** v2 sorted every batch by stay length, so each batch contained only short (or only long) stays.
v3 shuffles patients, sorts by length only inside random blocks of 50 batches, then shuffles the batches.

In [ ]:
LOG48 = math.log1p(48.)

def featurize(x, m, c):
    B, T, D = x.shape
    t = torch.arange(T, device=x.device).view(1, T, 1).expand(B, T, D)
    last = torch.cummax(torch.where(m > 0, t, torch.zeros_like(t)), dim=1)[0]
    ever = torch.cummax(m, dim=1)[0]
    xf = torch.gather(x, 1, last) * ever
    delta = torch.where(ever > 0, (t - last).float(), torch.full_like(x, 48.)).clamp(max=48.)
    dn = torch.log1p(delta) / LOG48
    v, l = slice(0, NV), slice(NV, D)
    rel = torch.stack([m[..., v].mean(-1), 1 - dn[..., v].mean(-1), ever[..., v].mean(-1),
                       m[..., l].mean(-1), 1 - dn[..., l].mean(-1), ever[..., l].mean(-1)], -1)
    avail = torch.stack([(rel[..., 2] > 0).float(), (rel[..., 5] > 0).float(), torch.ones_like(rel[..., 0])], -1)
    return {'vit': torch.cat([xf[..., v], m[..., v], dn[..., v]], -1),
            'lab': torch.cat([xf[..., l], m[..., l], dn[..., l]], -1),
            'xf': xf, 'm': m, 'delta': delta, 'dn': dn,
            'ctx': c, 'rel': rel, 'avail': avail, 'E': 0.5 * rel[..., 1] + 0.5 * rel[..., 5]}

def augment_mask(m):
    B, T, D = m.shape; dev = m.device; m = m.clone()
    drop_labs = (torch.rand(B, device=dev) < CFG['aug_p_labs']).float().view(B, 1, 1)
    m[..., NV:] = m[..., NV:] * (1 - drop_labs)
    sparse = torch.rand(B, device=dev) < CFG['aug_p_sparse_vitals']
    k = torch.randint(2, 5, (B,), device=dev).view(B, 1)
    keep_t = (torch.arange(T, device=dev).view(1, T) % k == 0) | (~sparse.view(B, 1))
    m[..., :NV] = m[..., :NV] * keep_t.unsqueeze(-1).float()
    rnd = (torch.rand(B, device=dev) < CFG['aug_p_random']).view(B, 1, 1)
    return m * (~((torch.rand(B, T, D, device=dev) < CFG['aug_random_drop']) & rnd)).float()

def make_batch(pids, M_src):
    L = pat.length.values[pids]; T = int(L.max()); B = len(pids)
    x = np.zeros((B, T, len(DYN)), np.float32); m = np.zeros((B, T, len(DYN)), np.float32)
    c = np.zeros((B, T, C_all.shape[1]), np.float32); y = np.zeros((B, T), np.float32); valid = np.zeros((B, T), bool)
    for i, p in enumerate(pids):
        s, e = pat.start.values[p], pat.end.values[p]; n = e - s
        x[i, :n], m[i, :n], c[i, :n], y[i, :n], valid[i, :n] = Z_all[s:e], M_src[s:e], C_all[s:e], y_all[s:e], True
    tt = lambda a: torch.from_numpy(a).to(device)
    return tt(x), tt(m), tt(c), tt(y), tt(valid)

def batches(pidx, bs, shuffle, rng=None):
    if not shuffle:
        order = pidx[np.argsort(pat.length.values[pidx], kind='stable')]
        return [order[i:i + bs] for i in range(0, len(order), bs)]
    perm, chunks = rng.permutation(pidx), []
    for i in range(0, len(perm), bs * 50):
        blk = perm[i:i + bs * 50]; blk = blk[np.argsort(pat.length.values[blk], kind='stable')]
        chunks += [blk[j:j + bs] for j in range(0, len(blk), bs)]
    rng.shuffle(chunks)
    return chunks
print('pipeline ready')

## 8 · Neural models
**Reliability-gated GRU-D** = v2's reliability-gated network + **GRU-D input decay** (Che et al., 2018): a missing
value decays from the last observation towards the population mean with a learned, per-variable rate
`γ = exp(−max(0, w·Δt + b))`, instead of being copied forward forever.

In [ ]:
class EarlyFusionGRU(nn.Module):
    def __init__(self, H=CFG['hidden'], p=CFG['dropout']):
        super().__init__(); self.p = p
        self.inp = nn.Sequential(nn.Linear(3 * NV + 3 * NL + 7 + 6, H), nn.ReLU())
        self.gru = nn.GRU(H, H, num_layers=2, batch_first=True, dropout=0.1)
        self.head = nn.Sequential(nn.Linear(H, H), nn.ReLU()); self.out = nn.Linear(H, 1)
    def encode(self, f):
        h, _ = self.gru(self.inp(torch.cat([f['vit'], f['lab'], f['ctx'], f['rel']], -1))); return h, {}
    def head_fwd(self, h, mc=False):
        on = self.training or mc
        return self.out(F.dropout(self.head(F.dropout(h, self.p, on)), self.p, on)).squeeze(-1)
    def forward(self, f, mc=False): return self.head_fwd(self.encode(f)[0], mc)

class ReliabilityGatedGRUD(nn.Module):
    def __init__(self, H=CFG['hidden'], p=CFG['dropout']):
        super().__init__(); self.p = p
        self.decay_w = nn.Parameter(torch.full((NV + NL,), 0.05)); self.decay_b = nn.Parameter(torch.zeros(NV + NL))
        self.vit_in = nn.Sequential(nn.Linear(3 * NV, H), nn.ReLU()); self.vit_gru = nn.GRU(H, H, batch_first=True)
        self.lab_in = nn.Sequential(nn.Linear(3 * NL, H), nn.ReLU()); self.lab_gru = nn.GRU(H, H, batch_first=True)
        self.ctx = nn.Sequential(nn.Linear(7, H), nn.ReLU(), nn.Linear(H, H), nn.ReLU())
        self.gate = nn.Sequential(nn.Linear(3 * H + 6, H), nn.ReLU(), nn.Linear(H, 3))
        nn.init.constant_(self.gate[-1].bias, 2.0)
        self.head = nn.Sequential(nn.Linear(3 * H + 6, H), nn.ReLU()); self.out = nn.Linear(H, 1)
    def encode(self, f):
        gamma = torch.exp(-F.relu(self.decay_w * f['delta'] + self.decay_b))          # GRU-D input decay
        xh = f['xf'] * (f['m'] + (1 - f['m']) * gamma)                               # decays to mean (=0 after z-score)
        v, l = slice(0, NV), slice(NV, NV + NL)
        hv, _ = self.vit_gru(self.vit_in(torch.cat([xh[..., v], f['m'][..., v], f['dn'][..., v]], -1)))
        hl, _ = self.lab_gru(self.lab_in(torch.cat([xh[..., l], f['m'][..., l], f['dn'][..., l]], -1)))
        hc = self.ctx(f['ctx'])
        w = torch.sigmoid(self.gate(torch.cat([hv, hl, hc, f['rel']], -1))) * f['avail']   # per-modality trust
        z = torch.cat([w[..., :1] * hv, w[..., 1:2] * hl, w[..., 2:] * hc, f['rel']], -1)
        return z, {'gates': w}
    def head_fwd(self, h, mc=False):
        on = self.training or mc
        return self.out(F.dropout(self.head(F.dropout(h, self.p, on)), self.p, on)).squeeze(-1)
    def forward(self, f, mc=False): return self.head_fwd(self.encode(f)[0], mc)

for cls in [EarlyFusionGRU, ReliabilityGatedGRUD]:
    print(f'{cls.__name__:>22}: {sum(p.numel() for p in cls().parameters()):,} parameters')

## 9 · Train the neural models

In [ ]:
@torch.no_grad()
def predict(model, S, M_src=M_all, mc=0, bs=256):
    model.eval(); n = len(S.rows)
    out = {'logit': np.zeros(n, np.float32), 'E': np.zeros(n, np.float32)}
    if mc: out['mc'] = np.zeros((mc, n), np.float32)
    for pids in batches(S.pidx, bs, False):
        x, m, c, y, valid = make_batch(pids, M_src)
        f = featurize(x, m, c); z, aux = model.encode(f)
        lg = model.head_fwd(z).cpu().numpy(); E = f['E'].cpu().numpy()
        mcl = torch.stack([model.head_fwd(z, mc=True) for _ in range(mc)]).cpu().numpy() if mc else None
        if 'gates' in aux and 'gates' not in out: out['gates'] = np.zeros((n, aux['gates'].shape[-1]), np.float32)
        for i, p in enumerate(pids):
            k = S.pos[p]; a, b = S.off[k], S.off[k + 1]; L = b - a
            out['logit'][a:b] = lg[i, :L]; out['E'][a:b] = E[i, :L]
            if mc: out['mc'][:, a:b] = mcl[:, i, :L]
            if 'gates' in aux: out['gates'][a:b] = aux['gates'][i, :L].cpu().numpy()
    return out

def train_model(key, model, augment, tr='train', va='val', tag='', seed=SEED):
    set_seed(seed); model = model.to(device); rng = np.random.default_rng(seed)
    Str, Sva = SI[tr], SI[va]
    steps = math.ceil(len(Str.pidx) / CFG['batch_size'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['lr'], total_steps=CFG['max_epochs'] * steps + 10, pct_start=0.1)
    pw = torch.tensor(float(np.sqrt((1 - Str.y.mean()) / Str.y.mean())), device=device)
    best, bad, hist, state = -1, 0, [], None
    for ep in range(CFG['max_epochs']):
        model.train(); t0 = time.time(); tot = cnt = 0
        for pids in batches(Str.pidx, CFG['batch_size'], True, rng):
            x, m, c, y, valid = make_batch(pids, M_all)
            if augment: m = augment_mask(m)
            logit = model(featurize(x, m, c))
            loss = (F.binary_cross_entropy_with_logits(logit, y, pos_weight=pw, reduction='none') * valid).sum() / valid.sum()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item() * valid.sum().item(); cnt += valid.sum().item()
        pv = predict(model, Sva)['logit']
        auprc, auroc = average_precision_score(Sva.y, pv), roc_auc_score(Sva.y, pv)
        hist.append({'epoch': ep + 1, 'loss': tot / cnt, 'val_AUPRC': auprc, 'val_AUROC': auroc})
        score = auroc + auprc; flag = ''
        if score > best:
            best, bad, flag = score, 0, '  *best*'
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else: bad += 1
        print(f'[{key}{tag}] ep {ep+1:2d} | loss/hour {tot/cnt:.4f} | val AUROC {auroc:.4f} AUPRC {auprc:.4f} | {time.time()-t0:.0f}s{flag}')
        if bad >= CFG['patience']: break
    model.load_state_dict(state); torch.save(state, os.path.join(OUT_DIR, f'{key}{tag}.pt'))
    pd.DataFrame(hist).to_csv(os.path.join(OUT_DIR, f'history_{key}{tag}.csv'), index=False)
    return model, hist

MODELS, HIST = {}, {}
for key, cls, aug in [('gru', EarlyFusionGRU, False), ('gru_aug', EarlyFusionGRU, True), ('rgn', ReliabilityGatedGRUD, True)]:
    MODELS[key], HIST[key] = train_model(key, cls(), aug)
    PRED[key] = {s: {'full': predict(MODELS[key], SI[s], mc=CFG['mc_passes'] if key == 'rgn' else 0)} for s in ['val', 'test']}

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for k, h in HIST.items():
    h = pd.DataFrame(h); ax[0].plot(h.epoch, h.loss, color=COLORS[k], label=MODEL_NAMES[k])
    ax[1].plot(h.epoch, h.val_AUROC, color=COLORS[k], label=MODEL_NAMES[k])
ax[0].set(title='Training loss (per hour)', xlabel='epoch'); ax[1].set(title='Validation AUROC', xlabel='epoch')
ax[1].legend(fontsize=8); savefig('02_training_curves.png'); plt.show()

## 10 · Hybrid risk engine + main results (held-out TEST)
**Hybrid** = logistic stacking of the LightGBM-ensemble logit and the RG-GRU-D logit, fitted on **validation**
(this also calibrates it). Its uncertainty samples pair each LightGBM member with an MC-dropout pass of the GRU-D.
All thresholds come from validation and are frozen before touching test.

In [ ]:
def fit_stack(lg_lgb, lg_rgn, y):
    lr = LogisticRegression(C=100, max_iter=1000).fit(np.c_[lg_lgb, lg_rgn], y)
    return float(lr.intercept_[0]), float(lr.coef_[0, 0]), float(lr.coef_[0, 1])
def hybrid_from(lgb_pred, rgn_pred, W):
    b0, b1, b2 = W
    out = {'logit': b0 + b1 * lgb_pred['logit'] + b2 * rgn_pred['logit'], 'E': rgn_pred['E']}
    if 'mc' in rgn_pred and 'members' in lgb_pred:
        K = len(lgb_pred['members'])
        out['mc'] = np.stack([b0 + b1 * lgb_pred['members'][j % K] + b2 * rgn_pred['mc'][j] for j in range(len(rgn_pred['mc']))])
    if 'gates' in rgn_pred: out['gates'] = rgn_pred['gates']
    return out

W_STACK = fit_stack(PRED['lgbm']['val']['full']['logit'], PRED['rgn']['val']['full']['logit'], SI['val'].y)
print('stacking weights (b0, w_lightgbm, w_rgn):', np.round(W_STACK, 3))
PRED['hybrid'] = {s: {'full': hybrid_from(PRED['lgbm'][s]['full'], PRED['rgn'][s]['full'], W_STACK)} for s in ['val', 'test']}

def prob(k, pred):
    return pred['score'] if k in RULES else sigmoid(AB[k][0] * pred['logit'] + AB[k][1])

ROWS = []
for k in MODEL_NAMES:
    if k in RULES: THR[k] = 2.0
    else:
        AB[k] = (1.0, 0.0) if k == 'hybrid' else fit_platt(PRED[k]['val']['full']['logit'], SI['val'].y)
        pv = prob(k, PRED[k]['val']['full']); THR[k] = best_threshold(SI['val'], pv); THR_LB[k] = low_burden_threshold(SI['val'], pv)
    pt = prob(k, PRED[k]['test']['full']); PRED[k]['test']['full']['p'] = pt; PRED[k]['val']['full']['p'] = prob(k, PRED[k]['val']['full'])
    r = evaluate(SI['test'], pt, THR[k], prob=k not in RULES); r['model'] = MODEL_NAMES[k]; ROWS.append(r)
RES = pd.DataFrame(ROWS).set_index('model'); RES.to_csv(os.path.join(OUT_DIR, 'main_results_test.csv'))
fmt = lambda r, m: f"{r[m]:.3f} [{r[m+'_lo']:.3f}–{r[m+'_hi']:.3f}]"
show_table(pd.DataFrame({'AUROC [95% CI]': [fmt(r, 'AUROC') for _, r in RES.iterrows()],
                         'AUPRC [95% CI]': [fmt(r, 'AUPRC') for _, r in RES.iterrows()],
                         'Utility': RES.utility.round(3), 'Patient AUROC': RES.patient_AUROC.round(3),
                         'Detected %': (100 * RES.detection_rate).round(1), 'Median lead (h)': RES.median_lead_h,
                         'False-alarm pts %': (100 * RES.false_alarm_patient_rate).round(1),
                         'Alert hours %': (100 * RES.alert_hour_rate).round(1), 'ECE': RES.ECE.round(4)}, index=RES.index))

# ---- operating points for the deployable systems ----
OP = []
St = SI['test']
for k in ['lgbm', 'hybrid']:
    for name, thr in [('utility-optimal', THR[k]), ('low-burden (≤5% alert hours)', THR_LB[k])]:
        p = PRED[k]['test']['full']['p']; s = alert_stats(St, p >= thr)
        OP.append({'model': MODEL_NAMES[k], 'operating point': name, 'threshold': round(thr, 4), 'utility': round(utility(St, p >= thr), 3),
                   'detected %': round(100 * s['detection_rate'], 1), 'median lead h': s['median_lead_h'],
                   'false-alarm pts %': round(100 * s['false_alarm_patient_rate'], 1),
                   'false alerts / pt-day': round(s['false_alerts_per_patient_day'], 3),
                   'alert precision %': round(100 * s['alert_precision_hour'], 1), 'alert hours %': round(100 * s['alert_hour_rate'], 1)})
OP = pd.DataFrame(OP); OP.to_csv(os.path.join(OUT_DIR, 'operating_points_test.csv'), index=False); show_table(OP)

yt = St.y
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for k in MODEL_NAMES:
    p = PRED[k]['test']['full']['p']; lw = 2.2 if k == 'hybrid' else 1.2
    fpr, tpr, _ = roc_curve(yt, p); pr, rc, _ = precision_recall_curve(yt, p)
    ax[0].plot(fpr, tpr, color=COLORS[k], lw=lw, label=f"{MODEL_NAMES[k]} ({RES.loc[MODEL_NAMES[k], 'AUROC']:.3f})")
    ax[1].plot(rc, pr, color=COLORS[k], lw=lw, label=f"{RES.loc[MODEL_NAMES[k], 'AUPRC']:.3f}")
ax[0].plot([0, 1], [0, 1], 'k:', lw=0.8); ax[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC (hour level)')
ax[1].axhline(yt.mean(), color='k', ls=':', lw=0.8); ax[1].set(xlabel='Recall', ylabel='Precision', title='Precision–Recall (hour level)', ylim=(0, 0.6))
ax[0].legend(fontsize=7, loc='lower right'); ax[1].legend(fontsize=7.5, title='AUPRC')
for lab, p, st_, col in [('RG-GRU-D, uncalibrated', sigmoid(PRED['rgn']['test']['full']['logit']), '--', COLORS['rgn']),
                         ('Hybrid (stacked = calibrated)', PRED['hybrid']['test']['full']['p'], '-', COLORS['hybrid'])]:
    ch = np.array_split(np.argsort(p), 12)
    ax[2].plot([p[c].mean() for c in ch], [yt[c].mean() for c in ch], st_, marker='o', color=col, label=lab)
mx = max(0.05, float(np.quantile(PRED['hybrid']['test']['full']['p'], 0.999)))
ax[2].plot([0, 1], [0, 1], 'k:', lw=0.8); ax[2].set(xlabel='Predicted risk', ylabel='Observed sepsis-label rate',
         title='Calibration (12 equal-mass bins)', xscale='log', yscale='log', xlim=(1e-3, 1), ylim=(1e-3, 1))
ax[2].legend(fontsize=8)
savefig('03_roc_pr_calibration.png'); plt.show()

## 11 · Reliability-aware decision layer (our contribution)
Signals per hour: calibrated risk **p**, predictive uncertainty **σ** (spread over ensemble × MC-dropout samples),
evidence score **E**.
```
if   p ≥ 4τ                               → ALERT    (risk so high that uncertainty doesn't matter)
elif p ≥ τ/2 and (σ ≥ σ* or E ≤ E*)       → REVIEW   = evidence-seeking: "obtain labs / bedside reassessment"
elif p ≥ τ                                → ALERT
else                                      → MONITOR
```
`τ` = utility-optimal threshold; `σ*` / `E*` = 90th / 10th percentile of validation hours with p ≥ τ/2.
Section 12 tests what REVIEW is actually worth by revealing the labs it asks for.

In [ ]:
def decide(p, sigma, E, P):
    grey = (p >= P['tau_low']) & (p < P['tau_high']); unreliable = (sigma >= P['sigma_thr']) | (E <= P['E_thr'])
    return np.where(grey & unreliable, 'REVIEW', np.where(p >= P['tau'], 'ALERT', 'MONITOR'))
def pred_sigma(k, pred):
    return sigmoid(AB[k][0] * pred['mc'] + AB[k][1]).std(0)

PROP = 'hybrid'
Vv = PRED[PROP]['val']['full']; pv, Ev, sv = Vv['p'], Vv['E'], pred_sigma(PROP, Vv)
tau = THR[PROP]; grey = pv >= tau / 2
if grey.sum() < 50: grey = np.ones_like(pv, bool)
POLICY = {'tau': tau, 'tau_low': tau / 2, 'tau_high': 4 * tau,
          'sigma_thr': float(np.quantile(sv[grey], 0.90)), 'E_thr': float(np.quantile(Ev[grey], 0.10))}
print('Policy thresholds (validation):', {k: float(f'{v:.4g}') for k, v in POLICY.items()})

Tt = PRED[PROP]['test']['full']; pt, Et, stt = Tt['p'], Tt['E'], pred_sigma(PROP, Tt)
act = decide(pt, stt, Et, POLICY)
base_alert, pol_alert, review = pt >= tau, act == 'ALERT', act == 'REVIEW'
prow = []
for name, a in [('Threshold only', base_alert), ('Policy: ALERT hours', pol_alert), ('Policy: ALERT + REVIEW hours', pol_alert | review)]:
    s = alert_stats(St, a); s['utility'] = utility(St, a); s['policy'] = name; prow.append(s)
POL = pd.DataFrame(prow).set_index('policy'); POL['review_hour_rate'] = [0., review.mean(), review.mean()]
POL.to_csv(os.path.join(OUT_DIR, 'policy_results_test.csv')); show_table(POL.round(4).T)
prev = {a: float(St.y[act == a].mean()) for a in ['MONITOR', 'REVIEW', 'ALERT']}
print('Sepsis-label prevalence by action:', {k: f'{100*v:.1f}%' for k, v in prev.items()}, f'| overall {100*St.y.mean():.1f}%')

# ---- selective prediction: which hours should the system NOT decide on its own? ----
rank = lambda a: np.argsort(np.argsort(a, kind='stable'), kind='stable') / len(a)
tloc = t_all[St.rows]
ABST = {'Random': np.random.default_rng(0).random(len(pt)),
        'Early ICU hours first (time baseline)': -tloc.astype(float),
        'Uncertainty σ only': stt,
        'Low evidence only (1−E)': 1 - Et,
        'Reliability-aware (σ + low evidence)': rank(stt) + rank(1 - Et)}
cols = ['#bbbbbb', '#9c755f', '#4e79a7', '#76b7b2', COLORS['hybrid']]
cov = np.linspace(0.5, 1.0, 21); SEL = {}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for (name, sc), col in zip(ABST.items(), cols):
    order = np.argsort(sc, kind='stable'); au, ap = [], []
    for c in cov:
        keep = order[:int(c * len(order))]; yk = St.y[keep]
        au.append(roc_auc_score(yk, pt[keep]) if 0 < yk.sum() < len(yk) else np.nan)
        ap.append(average_precision_score(yk, pt[keep]) / max(yk.mean(), 1e-9) if yk.sum() else np.nan)
    SEL[name] = {'coverage': cov.tolist(), 'AUROC': au, 'AUPRC_lift': ap}
    ax[0].plot(cov, au, marker='.', color=col, label=name); ax[1].plot(cov, ap, marker='.', color=col, label=name)
ax[0].set(xlabel='Coverage (fraction of hours decided automatically)', ylabel='AUROC on retained hours', title='Selective prediction — discrimination')
ax[1].set(xlabel='Coverage', ylabel='AUPRC ÷ prevalence (lift)', title='Selective prediction — precision lift')
ax[0].legend(fontsize=7.5); json.dump(SEL, open(os.path.join(OUT_DIR, 'selective_prediction.json'), 'w'))
savefig('04_selective_prediction.png'); plt.show()

## 12 · Stress tests + the value of REVIEW (evidence-seeking experiment)
Same trained models and thresholds; only the test-time evidence is degraded (labs removed, 4-hourly vitals,
50% vitals dropped, both). Then the key experiment: **during a total lab outage, every patient the policy sends to
REVIEW gets their labs back from that hour onward** (we have the real lab values, so this counterfactual is exact
for the labs that were actually drawn). If REVIEW is useful, this should recover much of the lost performance while
requesting labs for only a fraction of patients.

In [ ]:
def stress_mask(name, rng=np.random.default_rng(123)):
    Mk = M_all.copy()
    if 'labs_missing' in name: Mk[:, NV:] = False
    if 'vitals_q4h' in name: Mk[np.ix_((t_all % 4) != 0, np.arange(NV))] = False
    if 'vitals_drop50' in name: Mk[:, :NV] &= rng.random((N_ROWS, NV)) >= 0.5
    return Mk

def run_all_models(Mk, S=None, keys=None, mc=None):
    """Predictions of every model for split S under observation mask Mk."""
    S = S or St; keys = keys or list(MODEL_NAMES); out = {}
    Fx = lgb_features(S.rows, Mk)
    for k in keys:
        if k in RULES: out[k] = {'score': Fx[k].fillna(0).values.astype(np.float32)}
        elif k == 'lgbm':
            mem = lgb_members(LGB_MODELS, Fx); out[k] = {'logit': mem.mean(0), 'members': mem}
        elif k == 'hybrid': out[k] = hybrid_from(out['lgbm'], out['rgn'], W_STACK)
        else: out[k] = predict(MODELS[k], S, Mk, mc=(CFG['mc_passes'] if mc is None else mc) if k == 'rgn' else 0)
    return out

SCEN = ['full', 'labs_missing', 'vitals_q4h', 'vitals_drop50', 'labs_missing+vitals_q4h']
srows, SP = [], {}
for sc in SCEN:
    SP[sc] = {k: PRED[k]['test']['full'] for k in MODEL_NAMES} if sc == 'full' else run_all_models(stress_mask(sc))
    for k in MODEL_NAMES:
        p = prob(k, SP[sc][k])
        r = {'scenario': sc, 'model': MODEL_NAMES[k], 'key': k, 'AUROC': roc_auc_score(St.y, p),
             'AUPRC': average_precision_score(St.y, p), 'utility': utility(St, p >= THR[k])}
        r.update({kk: alert_stats(St, p >= THR[k])[kk] for kk in ['detection_rate', 'false_alarm_patient_rate']})
        if k == PROP:
            a = decide(p, pred_sigma(k, SP[sc][k]), SP[sc][k]['E'], POLICY)
            r.update({'review_hour_rate': float((a == 'REVIEW').mean()), 'policy_utility': utility(St, a == 'ALERT'),
                      'policy_utility_with_review': utility(St, a != 'MONITOR')})
        if k == 'rgn':
            r.update({f'gate_{g}': float(SP[sc][k]['gates'][:, i].mean()) for i, g in enumerate(['vitals', 'labs', 'context'])})
        srows.append(r)
    print('done:', sc)
STRESS = pd.DataFrame(srows)
full = STRESS[STRESS.scenario == 'full'].set_index('key')
STRESS['AUROC_drop'] = STRESS.apply(lambda r: r.AUROC - full.loc[r.key, 'AUROC'], axis=1)
STRESS.to_csv(os.path.join(OUT_DIR, 'stress_tests_test.csv'), index=False)
show_table(STRESS.pivot(index='scenario', columns='model', values='AUROC').loc[SCEN].round(3))

# ---- evidence-seeking experiment under a total lab outage ----
def first_hours(flags):
    """first local hour per patient where flags is True (or -1)."""
    return np.array([(np.flatnonzero(flags[St.off[i]:St.off[i + 1]])[:1].tolist() or [-1])[0] for i in range(len(St.pidx))])
def reveal_labs_from(first):
    Mk = stress_mask('labs_missing')
    for i, h in enumerate(first):
        if h < 0: continue
        s, e = pat.start.values[St.pidx[i]], pat.end.values[St.pidx[i]]
        Mk[s + h:e, NV:] = M_all[s + h:e, NV:]
    return Mk
out_pred = SP['labs_missing'][PROP]; p_out = prob(PROP, out_pred)
act_out = decide(p_out, pred_sigma(PROP, out_pred), out_pred['E'], POLICY)
TRIG = {'Lab outage + REVIEW requests labs (policy)': first_hours(act_out == 'REVIEW'),
        'Lab outage + labs requested whenever p ≥ τ/2 (risk-only trigger)': first_hours(p_out >= POLICY['tau_low'])}
erows = []
def add_row(name, pred, requested):
    p = prob(PROP, pred); a = decide(p, pred_sigma(PROP, pred), pred['E'], POLICY); s = alert_stats(St, p >= THR[PROP])
    erows.append({'setting': name, 'AUROC': round(roc_auc_score(St.y, p), 4), 'utility (threshold)': round(utility(St, p >= THR[PROP]), 3),
                  'utility (policy ALERT)': round(utility(St, a == 'ALERT'), 3), 'detected %': round(100 * s['detection_rate'], 1),
                  'false-alarm pts %': round(100 * s['false_alarm_patient_rate'], 1), 'patients with lab request %': requested})
add_row('All labs available (reference)', SP['full'][PROP], np.nan)
add_row('Lab outage, no action', out_pred, 0.0)
for name, first in TRIG.items():
    add_row(name, run_all_models(reveal_labs_from(first), keys=['lgbm', 'rgn', 'hybrid'])[PROP], round(100 * (first >= 0).mean(), 1))
EVID = pd.DataFrame(erows).set_index('setting'); EVID.to_csv(os.path.join(OUT_DIR, 'evidence_seeking_experiment.csv'))
lost = EVID.iloc[0]['utility (threshold)'] - EVID.iloc[1]['utility (threshold)']
rec = (EVID.iloc[2]['utility (threshold)'] - EVID.iloc[1]['utility (threshold)']) / lost if lost > 0 else np.nan
show_table(EVID); print(f'REVIEW-triggered labs recover {100*rec:.0f}% of the utility lost to the outage')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={'width_ratios': [2.2, 1]})
keys = [k for k in MODEL_NAMES if k not in RULES]; w = 0.8 / len(keys); xs = np.arange(len(SCEN))
for i, k in enumerate(keys):
    ax[0].bar(xs + i * w - 0.4 + w / 2, STRESS[STRESS.key == k].set_index('scenario').loc[SCEN, 'AUROC'], w, color=COLORS[k], label=MODEL_NAMES[k])
ax[0].set_xticks(xs); ax[0].set_xticklabels([s.replace('+', '\n+') for s in SCEN], fontsize=8)
sub = STRESS[STRESS.key.isin(keys)]
ax[0].set_ylim(max(0.5, sub.AUROC.min() - 0.03), min(1, sub.AUROC.max() + 0.02)); ax[0].set_ylabel('AUROC (test)')
ax[0].set_title('Robustness to missing / degraded evidence'); ax[0].legend(fontsize=7.5, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.16))
labels = ['labs\navailable', 'outage,\nno action', 'outage +\nREVIEW→labs', 'outage +\nrisk trigger']
ax[1].bar(range(len(EVID)), EVID['utility (threshold)'], color=['#8c8c8c', '#bab0ac', COLORS['hybrid'], '#4e79a7'])
for i, (u, q) in enumerate(zip(EVID['utility (threshold)'], EVID['patients with lab request %'])):
    ax[1].text(i, u + 0.005, f'{u:.3f}' + ('' if np.isnan(q) else f'\n{q:.0f}% req.'), ha='center', fontsize=7.5)
ax[1].set_xticks(range(len(EVID))); ax[1].set_xticklabels(labels, fontsize=7.5); ax[1].set_ylabel('PhysioNet utility (test)')
u_ = EVID['utility (threshold)']; ax[1].set_ylim(max(0, u_.min() - 0.1), u_.max() + 0.06)
ax[1].set_title('Value of REVIEW during a lab outage')
savefig('05_stress_and_evidence_seeking.png'); plt.show()

## 13 · External validation: train on hospital A → test on hospital B
Real-world value means working in a hospital the model has never seen. Tree models, the RG-GRU-D and the
hybrid are retrained using **only hospital A** patients (thresholds from A-validation) and tested on **all
hospital B patients**. (Normalisation statistics still come from the pooled training split — a minor, label-free
leak we report as a limitation.)

In [ ]:
if RUN_CROSS_HOSPITAL and len(SPLITS['B_all']) > 100 and len(SPLITS['A_train']) > 100:
    t0 = time.time()
    FA = {k: lgb_features(SI[k].rows, M_all) for k in ['A_train', 'A_val', 'A_test', 'B_all']}
    lgbA = train_lgb(FA['A_train'], SI['A_train'].y, FA['A_val'], SI['A_val'].y, [SEED])
    rgnA, _ = train_model('rgn', ReliabilityGatedGRUD(), True, 'A_train', 'A_val', tag='_hospA')
    PA = {}
    for k in ['A_val', 'A_test', 'B_all']:
        mem = lgb_members(lgbA, FA[k]); pl = {'logit': mem.mean(0), 'members': mem}; pr_ = predict(rgnA, SI[k])
        PA[k] = {'lgbm': pl, 'rgn': pr_}
    WA = fit_stack(PA['A_val']['lgbm']['logit'], PA['A_val']['rgn']['logit'], SI['A_val'].y)
    xrows = []
    for k, name in [('lgbm', MODEL_NAMES['lgbm']), ('rgn', MODEL_NAMES['rgn']), ('hybrid', MODEL_NAMES['hybrid']), ('sirs', MODEL_NAMES['sirs'])]:
        def pr(split):
            if k == 'sirs': return FA[split]['sirs'].fillna(0).values
            if k == 'hybrid': return sigmoid(hybrid_from(PA[split]['lgbm'], PA[split]['rgn'], WA)['logit'])
            ab = fit_platt(PA['A_val'][k]['logit'], SI['A_val'].y); return sigmoid(ab[0] * PA[split][k]['logit'] + ab[1])
        thr = 2.0 if k == 'sirs' else best_threshold(SI['A_val'], pr('A_val'))
        for split, lab in [('A_test', 'internal (hospital A test)'), ('B_all', 'external (hospital B, unseen)')]:
            p = pr(split); S = SI[split]; s = alert_stats(S, p >= thr)
            xrows.append({'model': name, 'test set': lab, 'AUROC': roc_auc_score(S.y, p), 'AUPRC': average_precision_score(S.y, p),
                          'utility': utility(S, p >= thr), 'detected %': 100 * s['detection_rate'], 'false-alarm pts %': 100 * s['false_alarm_patient_rate']})
    XH = pd.DataFrame(xrows); XH.to_csv(os.path.join(OUT_DIR, 'external_validation_A_to_B.csv'), index=False)
    show_table(XH.round(3)); print(f'cross-hospital section: {time.time()-t0:.0f}s')
    del FA
else:
    XH = None; print('Cross-hospital validation skipped (needs both hospitals).')

## 14 · Shortcut ablation: is the model using real physiology?
The v3 SHAP plot showed **ICULOS (hours in ICU)** as the largest driver of one alert, and features such as
`FiO2_dt = 999` ("never measured") carry information about **which tests clinicians chose to order**. Neither is
future leakage, but both are shortcuts reviewers ask about. We retrain:

* **LightGBM without time** (ICULOS, HospAdmTime removed)
* **LightGBM without time + measurement pattern** (also every `*_dt` "hours since measured" and `labs_seen`)
* **RG-GRU-D without time** (the two time columns of its context input set to 0; its masks/deltas are its
  missing-data mechanism and stay)
* **Hybrids** built from them, stacked on validation as before.

If performance holds, the shortcut concern is answered. If it drops, the ablated model is the honest headline.

In [ ]:
# ---- paper-safety toolkit: fast paired bootstrap over patients ----
def util_parts(S, alert):
    r, o = S.rows, S.off[:-1]
    obs  = np.add.reduceat(np.where(alert, U_POS[r], U_NEG[r]).astype(np.float64), o)
    best = np.add.reduceat(np.where(BEST[r], U_POS[r], U_NEG[r]).astype(np.float64), o)
    ina  = np.add.reduceat(U_NEG[r].astype(np.float64), o)
    return obs, best, ina
def util_from(parts, cnt):
    obs, best, ina = parts
    return (cnt @ obs - cnt @ ina) / (cnt @ best - cnt @ ina)

def paired_test(S, pa, pb, thr_a=None, thr_b=None, B=None, seed=0):
    # Paired bootstrap: resample the SAME patients for both models. Returns {metric: (diff, lo, hi, p)}.
    B = B or CFG['n_pair_boot']; rng = np.random.default_rng(seed); y = S.y; n = len(S.pidx)
    base = {'AUROC': roc_auc_score(y, pa) - roc_auc_score(y, pb),
            'AUPRC': average_precision_score(y, pa) - average_precision_score(y, pb)}
    ua = ub = None
    if thr_a is not None:
        ua, ub = util_parts(S, pa >= thr_a), util_parts(S, pb >= thr_b); one = np.ones(n)
        base['utility'] = util_from(ua, one) - util_from(ub, one)
    draws = {k: [] for k in base}
    for _ in range(B):
        cnt = np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64)
        w = np.repeat(cnt, S.lengths); m = w > 0; yy, ww = y[m], w[m]
        if yy.min() == yy.max(): continue
        draws['AUROC'].append(roc_auc_score(yy, pa[m], sample_weight=ww) - roc_auc_score(yy, pb[m], sample_weight=ww))
        draws['AUPRC'].append(average_precision_score(yy, pa[m], sample_weight=ww) - average_precision_score(yy, pb[m], sample_weight=ww))
        if ua is not None: draws['utility'].append(util_from(ua, cnt) - util_from(ub, cnt))
    out = {}
    for k, v in draws.items():
        v = np.asarray(v); lo, hi = np.percentile(v, [2.5, 97.5])
        out[k] = (base[k], lo, hi, min(1.0, 2 * min((v <= 0).mean(), (v >= 0).mean())))
    return out
def fmt_diff(t):
    d, lo, hi, p = t
    ps = f'< {1/CFG["n_pair_boot"]:.3f}' if p == 0 else f'{p:.3f}'
    return f'{d:+.3f} [{lo:+.3f}, {hi:+.3f}] (p {ps})'

def eval_logits(name, lv, lt, calibrated=False):
    ab = (1.0, 0.0) if calibrated else fit_platt(lv, SI['val'].y)
    pv_, pt_ = sigmoid(ab[0] * lv + ab[1]), sigmoid(ab[0] * lt + ab[1])
    thr = best_threshold(SI['val'], pv_)
    r = evaluate(St, pt_, thr); r['model'] = name
    return r, pt_, thr

ABLT = None
if RUN_ABLATION:
    t0 = time.time()
    Ftr_full = lgb_features(SI['train'].rows, M_all)
    TIME_COLS = ['ICULOS', 'HospAdmTime']
    MEAS_COLS = [c for c in Ftr_full.columns if c.endswith('_dt')] + ['labs_seen']
    ABL = {'no_time': TIME_COLS, 'no_time_no_meas': TIME_COLS + MEAS_COLS}
    abl_seeds = [SEED + i for i in range(min(3, CFG['lgb_seeds']))]
    ABLP = {}
    for name, drop in ABL.items():
        keep = [c for c in Ftr_full.columns if c not in drop]
        print(f'LightGBM {name}: {len(keep)} features')
        ms = train_lgb(Ftr_full[keep], SI['train'].y, FEAT['val'][keep], SI['val'].y, abl_seeds)
        ABLP['lgbm_' + name] = {s: lgb_members(ms, FEAT[s][keep]).mean(0) for s in ['val', 'test']}
        if name == 'no_time': LGB_NOTIME, KEEP_NOTIME = ms, keep
    del Ftr_full
    C_FULL = C_all
    try:
        C_all = C_FULL.copy(); C_all[:, 5:7] = 0.0            # hospital-admission time + ICU time columns
        rgn_nt, _ = train_model('rgn', ReliabilityGatedGRUD(), True, tag='_notime')
        ABLP['rgn_no_time'] = {s: predict(rgn_nt, SI[s])['logit'] for s in ['val', 'test']}
    finally:
        C_all = C_FULL
    for name in ABL:
        W = fit_stack(ABLP['lgbm_' + name]['val'], ABLP['rgn_no_time']['val'], SI['val'].y)
        ABLP['hybrid_' + name] = {s: W[0] + W[1] * ABLP['lgbm_' + name][s] + W[2] * ABLP['rgn_no_time'][s] for s in ['val', 'test']}
    rows, APROB = [], {}
    spec = [('Hybrid, all features (main model)', PRED['hybrid']['val']['full']['logit'], PRED['hybrid']['test']['full']['logit'], True),
            ('LightGBM, all features', PRED['lgbm']['val']['full']['logit'], PRED['lgbm']['test']['full']['logit'], False),
            ('RG-GRU-D, all features', PRED['rgn']['val']['full']['logit'], PRED['rgn']['test']['full']['logit'], False),
            ('LightGBM without time', ABLP['lgbm_no_time']['val'], ABLP['lgbm_no_time']['test'], False),
            ('LightGBM without time + measurement pattern', ABLP['lgbm_no_time_no_meas']['val'], ABLP['lgbm_no_time_no_meas']['test'], False),
            ('RG-GRU-D without time', ABLP['rgn_no_time']['val'], ABLP['rgn_no_time']['test'], False),
            ('Hybrid without time', ABLP['hybrid_no_time']['val'], ABLP['hybrid_no_time']['test'], True),
            ('Hybrid without time + measurement pattern', ABLP['hybrid_no_time_no_meas']['val'], ABLP['hybrid_no_time_no_meas']['test'], True)]
    for name, lv, lt, cal in spec:
        r, p_, thr_ = eval_logits(name, lv, lt, cal); rows.append(r); APROB[name] = (p_, thr_)
    ABLR = pd.DataFrame(rows).set_index('model'); ABLR.to_csv(os.path.join(OUT_DIR, 'v4_shortcut_ablation.csv'))
    fmt = lambda r, m: f"{r[m]:.3f} [{r[m+'_lo']:.3f}–{r[m+'_hi']:.3f}]"
    ABLT = pd.DataFrame({'AUROC [95% CI]': [fmt(r, 'AUROC') for _, r in ABLR.iterrows()], 'AUPRC [95% CI]': [fmt(r, 'AUPRC') for _, r in ABLR.iterrows()],
                         'Utility': ABLR.utility.round(3), 'Detected %': (100 * ABLR.detection_rate).round(1),
                         'False-alarm pts %': (100 * ABLR.false_alarm_patient_rate).round(1)}, index=ABLR.index)
    show_table(ABLT)
    pa_, ta_ = APROB['Hybrid, all features (main model)']; pb_, tb_ = APROB['Hybrid without time']
    t = paired_test(St, pa_, pb_, ta_, tb_)
    print('Main hybrid minus hybrid-without-time:', {k: fmt_diff(v) for k, v in t.items()})
    imp = pd.Series(np.mean([m.feature_importance('gain') for m in LGB_NOTIME], 0), index=KEEP_NOTIME).sort_values(ascending=False)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1.35, 1]})
    names = list(ABLR.index)[::-1]; ys = np.arange(len(names))
    for i, nm in enumerate(names):
        r = ABLR.loc[nm]; col = COLORS['hybrid'] if nm.startswith('Hybrid') else (COLORS['lgbm'] if nm.startswith('LightGBM') else COLORS['rgn'])
        ax[0].plot([r.AUROC_lo, r.AUROC_hi], [i, i], color=col, lw=2); ax[0].plot(r.AUROC, i, 'o', color=col, ms=7)
    ax[0].set_yticks(ys); ax[0].set_yticklabels(names, fontsize=8.5); ax[0].set_xlabel('Test AUROC with 95% CI')
    ax[0].set_title('Shortcut ablation'); ax[0].axvline(ABLR.loc['Hybrid, all features (main model)', 'AUROC'], color='k', ls=':', lw=0.8)
    top = imp.index[:12][::-1]; ax[1].barh(top, imp[top] / imp.sum() * 100, color=COLORS['lgbm'])
    ax[1].set_xlabel('% of total gain'); ax[1].set_title('LightGBM without time: top features')
    savefig('09_shortcut_ablation.png'); plt.show()
    print(f'shortcut ablation: {time.time()-t0:.0f}s')
else:
    print('Shortcut ablation skipped (RUN_ABLATION = False)')

## 15 · Five seeds per neural model + paired significance tests
Neural networks give slightly different results with each random seed. Every neural model is trained **5 times**
(seed 0 = the main run above) and reported as mean ± standard deviation. The hybrid is rebuilt with each seed's
RG-GRU-D. **Paired bootstrap**: the same resampled patients are scored by both models, 200 times; the 95% interval
of the difference and a two-sided bootstrap p-value tell us whether a gap is real.

In [ ]:
SEEDT = PAIRT = None
if RUN_SEEDS:
    t0 = time.time(); srows = []
    def seed_row(model, s, lv, lt, calibrated=False):
        ab = (1.0, 0.0) if calibrated else fit_platt(lv, SI['val'].y)
        pv_, pt_ = sigmoid(ab[0] * lv + ab[1]), sigmoid(ab[0] * lt + ab[1]); thr = best_threshold(SI['val'], pv_)
        srows.append({'model': MODEL_NAMES[model], 'seed': s, 'AUROC': roc_auc_score(St.y, pt_), 'AUPRC': average_precision_score(St.y, pt_),
                      'utility': utility(St, pt_ >= thr)})
    for k in ['gru', 'gru_aug', 'rgn']: seed_row(k, 0, PRED[k]['val']['full']['logit'], PRED[k]['test']['full']['logit'])
    seed_row('hybrid', 0, PRED['hybrid']['val']['full']['logit'], PRED['hybrid']['test']['full']['logit'], True)
    for s in range(1, CFG['n_seeds']):
        for k, cls, aug in [('gru', EarlyFusionGRU, False), ('gru_aug', EarlyFusionGRU, True), ('rgn', ReliabilityGatedGRUD, True)]:
            m, _ = train_model(k, cls(), aug, tag=f'_seed{s}', seed=SEED + s)
            lv, lt = predict(m, SI['val'])['logit'], predict(m, SI['test'])['logit']
            seed_row(k, s, lv, lt)
            if k == 'rgn':
                W = fit_stack(PRED['lgbm']['val']['full']['logit'], lv, SI['val'].y)
                seed_row('hybrid', s, W[0] + W[1] * PRED['lgbm']['val']['full']['logit'] + W[2] * lv,
                         W[0] + W[1] * PRED['lgbm']['test']['full']['logit'] + W[2] * lt, True)
            del m
            if device.type == 'cuda': torch.cuda.empty_cache()
    SEEDS = pd.DataFrame(srows); SEEDS.to_csv(os.path.join(OUT_DIR, 'v4_seed_runs.csv'), index=False)
    g = SEEDS.groupby('model')[['AUROC', 'AUPRC', 'utility']]
    SEEDT = pd.DataFrame({c: [f'{mu:.3f} ± {sd:.3f}' for mu, sd in zip(g.mean()[c], g.std(ddof=1)[c])] for c in ['AUROC', 'AUPRC', 'utility']}, index=g.mean().index)
    SEEDT['seeds'] = g.size(); SEEDT.loc[MODEL_NAMES['lgbm']] = [f"{RES.loc[MODEL_NAMES['lgbm'], c]:.3f} (5-seed ensemble)" for c in ['AUROC', 'AUPRC', 'utility']] + [1]
    print(f'Test metrics over {CFG["n_seeds"]} seeds (mean ± sd):'); show_table(SEEDT)

    # ---- paired tests: main-seed models ----
    prow = []
    def add_pair(label, setting, S, pa, pb, ta, tb):
        t = paired_test(S, pa, pb, ta, tb)
        prow.append({'comparison': label, 'test set': setting, **{f'Δ {k} [95% CI] (p)': fmt_diff(v) for k, v in t.items()}})
    P = lambda k: PRED[k]['test']['full']['p']
    for a, b in [('hybrid', 'lgbm'), ('rgn', 'gru'), ('rgn', 'gru_aug'), ('hybrid', 'rgn')]:
        add_pair(f'{MODEL_NAMES[a]} vs {MODEL_NAMES[b]}', 'internal test', St, P(a), P(b), THR[a], THR[b])
    for sc in ['labs_missing', 'labs_missing+vitals_q4h']:
        for a, b in [('hybrid', 'lgbm'), ('rgn', 'gru')]:
            add_pair(f'{MODEL_NAMES[a]} vs {MODEL_NAMES[b]}', f'internal, {sc}', St, prob(a, SP[sc][a]), prob(b, SP[sc][b]), THR[a], THR[b])
    if XH is not None:
        def ext_p(k, split):
            if k == 'hybrid': return sigmoid(hybrid_from(PA[split]['lgbm'], PA[split]['rgn'], WA)['logit'])
            ab = fit_platt(PA['A_val'][k]['logit'], SI['A_val'].y); return sigmoid(ab[0] * PA[split][k]['logit'] + ab[1])
        for a, b in [('hybrid', 'lgbm'), ('rgn', 'lgbm')]:
            ta, tb = best_threshold(SI['A_val'], ext_p(a, 'A_val')), best_threshold(SI['A_val'], ext_p(b, 'A_val'))
            add_pair(f'{MODEL_NAMES[a]} vs {MODEL_NAMES[b]}', 'external hospital B (trained on A)', SI['B_all'], ext_p(a, 'B_all'), ext_p(b, 'B_all'), ta, tb)
    PAIRT = pd.DataFrame(prow).set_index('comparison'); PAIRT.to_csv(os.path.join(OUT_DIR, 'v4_paired_tests.csv'))
    print('\nPaired bootstrap (difference = first model minus second; p < 0.05 and a CI excluding 0 = a real difference):')
    show_table(PAIRT); print(f'seeds + tests: {time.time()-t0:.0f}s')
else:
    print('Seed runs skipped (RUN_SEEDS = False)')

## 16 · Evidence-seeking: trigger efficiency curve
Section 12 showed that requesting labs during an outage recovers most of the lost utility. The fair question is
**how many lab requests each trigger needs**. We sweep each trigger's threshold and plot utility against the % of
patients who received a lab request:

* **Reliability-aware (REVIEW rule):** request when risk ≥ m·τ/2 (below the direct-ALERT level 4τ) **and** evidence is unreliable
* **Risk only:** request whenever risk ≥ m·τ/2
* **Random:** request for a random fraction of patients at a random hour

A curve that sits **higher and further left** is more efficient: more utility for fewer requests.

In [ ]:
TRIG = None
if RUN_TRIGGER_CURVE:
    t0 = time.time()
    tau = POLICY['tau']; sig_out = pred_sigma(PROP, out_pred); E_out = out_pred['E']
    unrel = (sig_out >= POLICY['sigma_thr']) | (E_out <= POLICY['E_thr'])
    u_all, u_none = EVID.iloc[0]['utility (threshold)'], EVID.iloc[1]['utility (threshold)']
    def eval_request(first, trigger, param):
        pr = run_all_models(reveal_labs_from(first), keys=['lgbm', 'rgn', 'hybrid'], mc=0)['hybrid']
        p = prob(PROP, pr); a = p >= THR[PROP]; s = alert_stats(St, a); u = utility(St, a)
        return {'trigger': trigger, 'setting': param, 'patients requested %': round(100 * (first >= 0).mean(), 1),
                'utility': round(u, 4), 'lost utility recovered %': round(100 * (u - u_none) / max(u_all - u_none, 1e-9), 1),
                'AUROC': round(roc_auc_score(St.y, p), 4), 'detected %': round(100 * s['detection_rate'], 1),
                'false-alarm pts %': round(100 * s['false_alarm_patient_rate'], 1)}
    crow = []
    for m in CFG['trigger_mults']:
        tl = m * tau / 2
        crow.append(eval_request(first_hours((p_out >= tl) & (p_out < POLICY['tau_high']) & unrel), 'Reliability-aware (REVIEW rule)', f'm={m}'))
        crow.append(eval_request(first_hours(p_out >= tl), 'Risk only', f'm={m}'))
        print('done m =', m)
    rng_t = np.random.default_rng(7)
    for f in CFG['random_fracs']:
        first = np.full(len(St.pidx), -1); pick = rng_t.random(len(St.pidx)) < f
        first[pick] = (rng_t.random(pick.sum()) * St.lengths[pick]).astype(int)
        crow.append(eval_request(first, 'Random patients', f'{int(100*f)}%'))
    TRIG = pd.DataFrame(crow); TRIG.to_csv(os.path.join(OUT_DIR, 'v4_trigger_efficiency.csv'), index=False); show_table(TRIG)
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    for trig, col, mk in [('Reliability-aware (REVIEW rule)', COLORS['hybrid'], 'o'), ('Risk only', COLORS['gru'], 's'), ('Random patients', '#8c8c8c', '^')]:
        d = TRIG[TRIG.trigger == trig].sort_values('patients requested %')
        ax.plot(d['patients requested %'], d['utility'], marker=mk, color=col, label=trig)
    ax.axhline(u_all, color='k', ls='--', lw=1); ax.axhline(u_none, color='#bab0ac', ls='--', lw=1)
    for yv_, lab_ in [(u_all, 'all labs available'), (u_none, 'lab outage, no requests')]:
        ax.annotate(lab_, xy=(0.01, yv_), xycoords=('axes fraction', 'data'), xytext=(0, 3), textcoords='offset points', fontsize=8)
    ax.set_xlabel('% of patients who received a lab request'); ax.set_ylabel('PhysioNet utility (test)')
    ax.set_title('Evidence-seeking efficiency during a lab outage'); ax.legend(fontsize=8.5, loc='lower right'); ax.set_xlim(0, 100)
    savefig('10_trigger_efficiency.png'); plt.show(); print(f'trigger curve: {time.time()-t0:.0f}s')
else:
    print('Trigger curve skipped (RUN_TRIGGER_CURVE = False)')

## 17 · Selective-prediction diagnostics: where do deferred hours go?
Deferring hours can look good simply by removing the hard or positive cases. For each deferral score and coverage
we report the sepsis prevalence among **kept** and **deferred** hours, the share of all sepsis hours that were
deferred (these go to a clinician, so a high share is not hidden, but it must be visible), and a paired bootstrap
comparing the reliability-aware score with the "early ICU hours" time baseline.

In [ ]:
SELD = SELP = None
if RUN_SELECTIVE_DIAG:
    drow = []; n_pos = St.y.sum()
    keeps = {}
    for name, sc in ABST.items():
        order = np.argsort(sc, kind='stable')
        for c in [0.5, 0.6, 0.7, 0.8, 0.9]:
            keep = np.zeros(len(order), bool); keep[order[:int(c * len(order))]] = True; keeps[(name, c)] = keep
            yk, yd = St.y[keep], St.y[~keep]
            drow.append({'deferral score': name, 'coverage': c,
                         'AUROC kept': roc_auc_score(yk, pt[keep]) if 0 < yk.sum() < len(yk) else np.nan,
                         'prevalence kept %': 100 * yk.mean(), 'prevalence deferred %': 100 * yd.mean() if len(yd) else np.nan,
                         'sepsis hours deferred %': 100 * yd.sum() / n_pos,
                         'patients with a deferred hour %': 100 * np.maximum.reduceat((~keep).astype(np.int8), St.off[:-1]).mean()})
    SELD = pd.DataFrame(drow).round(3); SELD.to_csv(os.path.join(OUT_DIR, 'v4_selective_diagnostics.csv'), index=False)
    show_table(SELD[SELD.coverage.isin([0.6, 0.8])])
    rel, tb = 'Reliability-aware (σ + low evidence)', 'Early ICU hours first (time baseline)'
    prow, rng_s = [], np.random.default_rng(3)
    for c in [0.6, 0.8]:
        ka, kb = keeps[(rel, c)], keeps[(tb, c)]; diffs = []
        base = roc_auc_score(St.y[ka], pt[ka]) - roc_auc_score(St.y[kb], pt[kb])
        n = len(St.pidx)
        for _ in range(CFG['n_pair_boot']):
            w = np.repeat(np.bincount(rng_s.integers(0, n, n), minlength=n), St.lengths).astype(np.float64)
            ma, mb = ka & (w > 0), kb & (w > 0)
            if St.y[ma].min() == St.y[ma].max() or St.y[mb].min() == St.y[mb].max(): continue
            diffs.append(roc_auc_score(St.y[ma], pt[ma], sample_weight=w[ma]) - roc_auc_score(St.y[mb], pt[mb], sample_weight=w[mb]))
        d = np.asarray(diffs); lo, hi = np.percentile(d, [2.5, 97.5]); p = min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()))
        prow.append({'coverage': c, 'Δ AUROC kept (reliability − time) [95% CI] (p)': fmt_diff((base, lo, hi, p))})
    SELP = pd.DataFrame(prow).set_index('coverage'); show_table(SELP)
else:
    print('Selective diagnostics skipped (RUN_SELECTIVE_DIAG = False)')

## 18 · Conformal REVIEW: thresholds with a target miss rate
Instead of hand-picked cut-offs, **class-conditional (Mondrian) split conformal prediction** sets two thresholds
on the validation hours for a chosen error level α:

* include "sepsis" in the prediction set if risk ≥ t₁, where t₁ is chosen so that at most α of true sepsis hours fall below it
* include "no sepsis" if risk ≤ t₀, where t₀ is chosen so that at most α of non-sepsis hours lie above it

Set = {sepsis} → **ALERT**, {no sepsis} → **MONITOR**, both or neither → **REVIEW**. Hours from the same patient
are correlated, so the guarantee is approximate; the empirical miss rate on test patients is reported next to α.

In [ ]:
CONF = None
if RUN_CONFORMAL:
    yv, pv_c, pt_c = SI['val'].y, PRED[PROP]['val']['full']['p'], PRED[PROP]['test']['full']['p']
    def cq(scores, alpha):
        n = len(scores); lvl = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
        return float(np.quantile(scores, lvl, method='higher'))
    crow = []; pos = St.y == 1
    for a in CFG['conformal_alphas']:
        q1, q0 = cq(1 - pv_c[yv == 1], a), cq(pv_c[yv == 0], a)
        has1, has0 = (1 - pt_c) <= q1, pt_c <= q0
        act_c = np.where(has1 & ~has0, 'ALERT', np.where(has0 & ~has1, 'MONITOR', 'REVIEW'))
        s = alert_stats(St, act_c != 'MONITOR')
        crow.append({'α (target)': a, 'sepsis hours dismissed as MONITOR %': 100 * (act_c[pos] == 'MONITOR').mean(),
                     'non-sepsis hours in ALERT %': 100 * (act_c[~pos] == 'ALERT').mean(),
                     'ALERT hours %': 100 * (act_c == 'ALERT').mean(), 'REVIEW hours %': 100 * (act_c == 'REVIEW').mean(),
                     'patients flagged (ALERT or REVIEW) %': 100 * s['detection_rate'],
                     'utility (ALERT only)': utility(St, act_c == 'ALERT'), 'utility (ALERT + REVIEW)': utility(St, act_c != 'MONITOR')})
    CONF = pd.DataFrame(crow).round(3).set_index('α (target)'); CONF.to_csv(os.path.join(OUT_DIR, 'v4_conformal_review.csv')); show_table(CONF)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8)); al = np.array(CFG['conformal_alphas'])
    ax[0].plot(al * 100, al * 100, 'k:', lw=1, label='target (α)'); ax[0].plot(al * 100, CONF['sepsis hours dismissed as MONITOR %'], 'o-', color=COLORS['hybrid'], label='observed on test')
    ax[0].set(xlabel='target miss rate α (%)', ylabel='sepsis hours dismissed (%)', title='Does the miss-rate target hold?'); ax[0].legend(fontsize=8)
    ax[1].plot(al * 100, CONF['REVIEW hours %'], 'o-', color='#f2b701', label='REVIEW'); ax[1].plot(al * 100, CONF['ALERT hours %'], 's-', color=COLORS['hybrid'], label='ALERT')
    ax[1].set(xlabel='target miss rate α (%)', ylabel='% of hours', title='Price of a stricter guarantee'); ax[1].legend(fontsize=8)
    savefig('11_conformal_review.png'); plt.show()
else:
    print('Conformal REVIEW skipped (RUN_CONFORMAL = False)')

## 19 · Irrelevant-evidence control: does the gate ignore useless evidence?
We give the reliability-gated network a **fourth stream of pure random noise** (16 numbers per patient, constant over
the stay, like an unrelated image). It carries no information about sepsis by construction. If the gate works as
claimed, the noise stream should receive **lower trust than the real streams**, and discrimination should not drop.
This is a control experiment on real patients; the noise is never presented as data.

In [ ]:
IRR = None
if RUN_IRRELEVANT:
    class RGGRUD_WithNoise(ReliabilityGatedGRUD):
        def __init__(self, n_noise, H=CFG['hidden'], p=CFG['dropout']):
            super().__init__(H, p)
            self.noise_in = nn.Sequential(nn.Linear(n_noise, H), nn.ReLU(), nn.Linear(H, H), nn.ReLU())
            self.gate = nn.Sequential(nn.Linear(4 * H + 6, H), nn.ReLU(), nn.Linear(H, 4)); nn.init.constant_(self.gate[-1].bias, 2.0)
            self.head = nn.Sequential(nn.Linear(4 * H + 6, H), nn.ReLU())
        def encode(self, f):
            c = f['ctx']
            gamma = torch.exp(-F.relu(self.decay_w * f['delta'] + self.decay_b))
            xh = f['xf'] * (f['m'] + (1 - f['m']) * gamma)
            v, l = slice(0, NV), slice(NV, NV + NL)
            hv, _ = self.vit_gru(self.vit_in(torch.cat([xh[..., v], f['m'][..., v], f['dn'][..., v]], -1)))
            hl, _ = self.lab_gru(self.lab_in(torch.cat([xh[..., l], f['m'][..., l], f['dn'][..., l]], -1)))
            hc, hn = self.ctx(c[..., :7]), self.noise_in(c[..., 7:])
            avail = torch.cat([f['avail'], torch.ones_like(f['avail'][..., :1])], -1)
            w = torch.sigmoid(self.gate(torch.cat([hv, hl, hc, hn, f['rel']], -1))) * avail
            z = torch.cat([w[..., :1] * hv, w[..., 1:2] * hl, w[..., 2:3] * hc, w[..., 3:] * hn, f['rel']], -1)
            return z, {'gates': w}
    N_NOISE = 16
    noise_rows = np.repeat(np.random.default_rng(11).normal(size=(N_PAT, N_NOISE)).astype(np.float32), pat.length.values, axis=0)
    C_FULL = C_all
    try:
        C_all = np.hstack([C_FULL, noise_rows]).astype(np.float32)
        rgn_noise, _ = train_model('rgn', RGGRUD_WithNoise(N_NOISE), True, tag='_noise')
        pn = predict(rgn_noise, St); pn_val = predict(rgn_noise, SI['val'])
    finally:
        C_all = C_FULL
    del noise_rows
    g = pn['gates']; g_main = PRED['rgn']['test']['full']['gates']
    IRR = pd.DataFrame({'mean trust (with noise stream)': g.mean(0).tolist(),
                        'mean trust (main model)': g_main.mean(0).tolist() + [np.nan]},
                       index=['vitals', 'labs', 'context', 'NOISE (irrelevant)']).round(3)
    ab = fit_platt(pn_val['logit'], SI['val'].y); p_noise = sigmoid(ab[0] * pn['logit'] + ab[1])
    t = paired_test(St, p_noise, PRED['rgn']['test']['full']['p'])
    IRR.loc['test AUROC'] = [roc_auc_score(St.y, p_noise), RES.loc[MODEL_NAMES['rgn'], 'AUROC']]
    IRR.to_csv(os.path.join(OUT_DIR, 'v4_irrelevant_evidence.csv')); show_table(IRR)
    print('AUROC with noise stream minus main RG-GRU-D:', fmt_diff(t['AUROC']))
    lowest_real = g.mean(0)[:3].min()
    print('RESULT:', 'the gate gives the noise stream LESS trust than every real stream.' if g.mean(0)[3] < lowest_real
          else 'the gate does NOT clearly down-weight the noise stream; report this honestly (the head may be ignoring it instead).')
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for i, (nm, col) in enumerate([('vitals', '#4e79a7'), ('labs', '#f28e2b'), ('context', '#9c755f'), ('noise', '#8c8c8c')]):
        ax.hist(g[:, i], bins=40, range=(0, 1), alpha=0.55, color=col, label=nm, density=True)
    ax.set(xlabel='trust weight assigned by the gate (test hours)', ylabel='density', title='Irrelevant-evidence control'); ax.legend(fontsize=8)
    savefig('12_irrelevant_evidence_gate.png'); plt.show()
else:
    print('Irrelevant-evidence control skipped (RUN_IRRELEVANT = False)')

## 20 · Real-time monitor + live demo with evidence-seeking
`SepsisMonitor.update(one_hour_of_raw_data)` runs the full hybrid system (LightGBM ensemble + RG-GRU-D + decision
layer) and returns risk, uncertainty band, evidence, modality trust and the action. The replay uses a **held-out
septic test patient** (chosen automatically as the clearest pre-onset rise — an illustrative case, not a
performance claim) and three versions of their stay: all evidence, a lab outage with no action, and a lab outage
where a REVIEW makes the labs come back one hour later. Explanations are exact **TreeSHAP** values.

In [ ]:
class SepsisMonitor:
    def __init__(self, lgb_models, rgn, W, policy, norm=NORM, mc=CFG['mc_passes']):
        self.lgb, self.rgn, self.W, self.P, self.norm, self.mc = lgb_models, rgn.eval(), W, policy, norm, mc
        self.reset()
    def reset(self): self.X, self.D = [], []
    @torch.no_grad()
    def update(self, row):
        self.X.append([row.get(v, np.nan) for v in DYN]); self.D.append([row.get(v, np.nan) for v in DEMO])
        Xr, Dm = np.array(self.X, np.float32), np.array(self.D, np.float32); n = len(Xr)
        Z, M, C = to_model_inputs(Xr, Dm, self.norm)
        tt = lambda a: torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))[None].to(device)
        f = featurize(tt(Z), tt(M), tt(C)); z, aux = self.rgn.encode(f); zl = z[:, -1:]
        rg = self.rgn.head_fwd(zl).item()
        rg_mc = torch.stack([self.rgn.head_fwd(zl, mc=True) for _ in range(self.mc)]).flatten().cpu().numpy()
        Fh = lgb_feats_from(Xr, ~np.isnan(Xr), np.zeros(n, int), np.arange(n), Dm).iloc[[-1]]
        lg = np.array([m.predict(Fh, raw_score=True, num_iteration=m.best_iteration)[0] for m in self.lgb])
        b0, b1, b2 = self.W; K = len(lg)
        p = float(sigmoid(b0 + b1 * lg.mean() + b2 * rg))
        samp = sigmoid(np.array([b0 + b1 * lg[j % K] + b2 * rg_mc[j] for j in range(self.mc)]))
        E = float(f['E'][0, -1]); g = aux['gates'][0, -1].cpu().numpy()
        a = decide(np.array([p]), np.array([samp.std()]), np.array([E]), self.P)[0]
        return {'risk': p, 'risk_lo': min(p, float(np.percentile(samp, 5))), 'risk_hi': max(p, float(np.percentile(samp, 95))),
                'sigma': float(samp.std()), 'evidence': E, 'trust_vitals': g[0], 'trust_labs': g[1], 'trust_context': g[2], 'action': a}

mon = SepsisMonitor(LGB_MODELS, MODELS['rgn'], W_STACK, POLICY)
pT = PRED[PROP]['test']['full']['p']; cands = []
for i, pid in enumerate(St.pidx):
    if not St.septic[i]: continue
    y = St.y[St.off[i]:St.off[i + 1]]; ts = int(np.argmax(y)) + 6
    if ts < 24 or len(y) < ts + 3: continue
    pp = pT[St.off[i]:St.off[i + 1]]
    pre, peak = pp[:ts - 18].mean(), pp[ts - 12:ts + 1].max()
    if peak >= POLICY['tau'] and pre < POLICY['tau']: cands.append((peak - pre, pid, ts))
cands.sort(reverse=True)
if not cands:   # fallback: any septic test patient
    i = int(np.flatnonzero(St.septic)[0]); cands = [(0, St.pidx[i], int(np.argmax(St.y[St.off[i]:St.off[i + 1]])) + 6)]
_, DEMO_PID, TS = cands[0]
s0, e0 = pat.start.values[DEMO_PID], pat.end.values[DEMO_PID]; N_H = e0 - s0
OUTAGE = max(2, TS - 20)
print(f'Demo patient {pat.patient_id.values[DEMO_PID]} | stay {N_H} h | sepsis onset at hour {TS} | lab outage from hour {OUTAGE}')

def replay(outage=None, seek=False):
    mon.reset(); log, lat, back = [], [], None
    for t in range(N_H):
        row = dict(zip(DYN + DEMO, np.r_[X_raw[s0 + t], D_raw[s0 + t]]))
        hidden = outage is not None and t >= outage and (back is None or t < back)
        if hidden: row.update({v: np.nan for v in LABS})
        t1 = time.perf_counter(); o = mon.update(row); lat.append(time.perf_counter() - t1)
        o['hour'], o['labs_hidden'], o['lab_request'] = t, hidden, False
        if seek and hidden and o['action'] == 'REVIEW' and back is None: back, o['lab_request'] = t + 1, True
        log.append(o)
    return pd.DataFrame(log), np.array(lat)
CLEAN, lat = replay(); NOACT, _ = replay(OUTAGE); SEEK, _ = replay(OUTAGE, seek=True)
req = SEEK.hour[SEEK.lab_request].tolist()
print(f'Real-time latency: {1000*lat.mean():.1f} ms per hourly update (LightGBM×{len(LGB_MODELS)} + GRU-D×{CFG["mc_passes"]} MC passes, {device})')
print('Lab request (REVIEW) at hour:', req or 'none')
for _, r in SEEK.iloc[max(0, TS - 16):min(N_H, TS + 3)].iterrows():
    print(f"hour {int(r.hour):3d} | risk {100*r.risk:5.1f}% [{100*r.risk_lo:4.1f}–{100*r.risk_hi:4.1f}] | E {r.evidence:.2f} | "
          f"trust V/L/C {r.trust_vitals:.2f}/{r.trust_labs:.2f}/{r.trust_context:.2f} | {'labs hidden' if r.labs_hidden else 'labs ok    '} | {r.action}"
          + ('  ← labs requested' if r.lab_request else ''))

hrs = CLEAN.hour.values
fig, ax = plt.subplots(4, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [1.2, 1.7, 0.35, 1]})
for v, col in [('HR', '#e15759'), ('MAP', '#4e79a7'), ('Resp', '#59a14f'), ('Temp', '#f28e2b')]:
    j = DYN.index(v); vals = pd.Series(X_raw[s0:e0, j]).ffill().values
    ax[0].plot(hrs, (vals - NORM['mu'][j]) / NORM['sd'][j], color=col, label=v)
ax[0].set_ylabel('z-score'); ax[0].legend(ncol=4, fontsize=8, loc='upper left')
ax[0].set_title('Adaptive sepsis early warning — live replay of a held-out test patient')
ax[1].fill_between(hrs, 100 * SEEK.risk_lo, 100 * SEEK.risk_hi, color=COLORS['hybrid'], alpha=0.18, label='uncertainty band (5–95%)')
ax[1].plot(hrs, 100 * CLEAN.risk, 'k--', lw=1, label='all evidence')
ax[1].plot(hrs, 100 * NOACT.risk, color='#8c8c8c', lw=1.3, label='lab outage, no action')
ax[1].plot(hrs, 100 * SEEK.risk, color=COLORS['hybrid'], lw=2.2, label='lab outage + REVIEW → labs')
ax[1].axhline(100 * POLICY['tau'], color='k', ls=':', lw=1, label='alert threshold τ')
ax[1].set_ylabel('sepsis risk (%)'); ax[1].legend(fontsize=8, loc='upper left')
cmap = {'MONITOR': '#59a14f', 'ALERT': '#e15759', 'REVIEW': '#f2b701'}
ax[2].bar(hrs, np.ones(len(hrs)), width=1.0, color=[cmap[a] for a in SEEK.action]); ax[2].set_yticks([])
ax[2].set_ylabel('action', rotation=0, ha='right', va='center')
from matplotlib.patches import Patch
ax[2].legend(handles=[Patch(color=c, label=k) for k, c in cmap.items()], fontsize=8, loc='center left', bbox_to_anchor=(1.0, 0.5))
for col, c, lab in [('trust_vitals', '#4e79a7', 'trust: vitals'), ('trust_labs', '#f28e2b', 'trust: labs'), ('trust_context', '#9c755f', 'trust: context')]:
    ax[3].plot(hrs, SEEK[col], color=c, lw=2, label=lab)
ax[3].plot(hrs, SEEK.evidence, 'k--', lw=1.3, label='evidence E'); ax[3].set_ylim(-0.02, 1.02)
ax[3].set_ylabel('trust / evidence'); ax[3].set_xlabel('hours since ICU admission'); ax[3].legend(ncol=4, fontsize=8, loc='lower left')
hid = SEEK.hour[SEEK.labs_hidden]
for a in ax:
    a.axvline(TS, color='#b07aa1', lw=1.5)
    if len(hid): a.axvspan(hid.min() - 0.5, hid.max() + 0.5, color='grey', alpha=0.10)
    for h in req: a.axvline(h, color='#f2b701', lw=1.5, ls='--')
ax[1].text(TS + 0.3, ax[1].get_ylim()[1] * 0.92, 'sepsis onset', color='#b07aa1', fontsize=8)
if req: ax[1].text(req[0] + 0.3, ax[1].get_ylim()[1] * 0.80, 'REVIEW → labs requested', color='#b58900', fontsize=8)
savefig('06_live_dashboard.png'); plt.show()
SEEK.to_csv(os.path.join(OUT_DIR, 'demo_live_log.csv'), index=False)

# ---- exact TreeSHAP explanation at the first ALERT before onset (all evidence) ----
al = np.flatnonzero((CLEAN.action.values == 'ALERT')[:TS + 1])
t_star = int(al[0]) if len(al) else int(np.argmax(CLEAN.risk.values[:TS + 1]))
Fp = lgb_feats_from(X_raw[s0:e0], M_all[s0:e0], np.zeros(N_H, int), np.arange(N_H), D_raw[s0:e0]).iloc[[t_star]]
shap = np.mean([m.predict(Fp, pred_contrib=True, num_iteration=m.best_iteration)[0, :-1] for m in LGB_MODELS], 0)
top = np.argsort(-np.abs(shap))[:12][::-1]
plt.figure(figsize=(8, 4))
plt.barh([f'{Fp.columns[i]} = {Fp.iloc[0, i]:.3g}' for i in top], shap[top], color=['#e15759' if v > 0 else '#4e79a7' for v in shap[top]])
plt.axvline(0, color='k', lw=0.8); plt.xlabel('TreeSHAP contribution to risk (log-odds)  red = raises risk')
plt.title(f'Why did the system alert at hour {t_star}? (onset at hour {TS})'); savefig('07_shap_explanation.png'); plt.show()

## 21 · Save everything + slide-ready summary

In [ ]:
bundle = {'rgn_state_dict': MODELS['rgn'].state_dict(), 'lightgbm_files': [f'lightgbm_seed{i}.txt' for i in range(len(LGB_MODELS))],
          'stack_weights': W_STACK, 'policy': POLICY, 'cfg': CFG,
          'norm': {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in NORM.items()},
          'variables': {'vitals': VITALS, 'labs': LABS, 'demo': DEMO}}
torch.save(bundle, os.path.join(OUT_DIR, 'sepsis_monitor_bundle.pt'))
json.dump({'CFG': CFG, 'thresholds': THR, 'low_burden_thresholds': THR_LB, 'platt': AB, 'stack': W_STACK, 'policy': POLICY, 'seed': SEED},
          open(os.path.join(OUT_DIR, 'run_config.json'), 'w'), indent=2, default=float)
def md_table(df):
    try: return df.to_markdown()
    except Exception: return df.to_string()
lines = ['# Results summary (PhysioNet 2019, held-out test patients)', '',
         f"Patients: {summary['patients']:,} ({summary['septic_patient_pct']}% septic) | test patients: {len(St.pidx):,}", '',
         '| Model | AUROC [95% CI] | AUPRC [95% CI] | Utility | Detected | Median lead | False-alarm pts |', '|---|---|---|---|---|---|---|']
for m, r in RES.iterrows():
    lines.append(f"| {m} | {fmt(r, 'AUROC')} | {fmt(r, 'AUPRC')} | {r.utility:.3f} | {100*r.detection_rate:.0f}% | {r.median_lead_h:.0f} h | {100*r.false_alarm_patient_rate:.0f}% |")
lines += ['', '## Operating points', md_table(OP.set_index('model')), '', '## Decision policy (hybrid)', md_table(POL.round(3)),
          '', '## Stress tests (AUROC)', md_table(STRESS.pivot(index='scenario', columns='model', values='AUROC').loc[SCEN].round(3)),
          '', '## Evidence-seeking experiment (lab outage)', md_table(EVID)]
if XH is not None: lines += ['', '## External validation (train A → test B)', md_table(XH.round(3).set_index('model'))]
lines += ['', f'Real-time latency: {1000*lat.mean():.1f} ms per hourly update']
for title, tbl in [('v4 · Shortcut ablation', ABLT), ('v4 · Five-seed results (mean ± sd)', SEEDT), ('v4 · Paired bootstrap tests', PAIRT),
                   ('v4 · Trigger efficiency (lab outage)', TRIG.set_index('trigger') if TRIG is not None else None),
                   ('v4 · Selective-prediction diagnostics (coverage 0.6 / 0.8)', SELD[SELD.coverage.isin([0.6, 0.8])].set_index('deferral score') if SELD is not None else None),
                   ('v4 · Selective prediction: reliability vs time baseline', SELP), ('v4 · Conformal REVIEW', CONF), ('v4 · Irrelevant-evidence control', IRR)]:
    if tbl is not None: lines += ['', '## ' + title, md_table(tbl)]
open(os.path.join(OUT_DIR, 'SLIDE_SUMMARY.md'), 'w').write('\n'.join(lines))
print('\n'.join(lines)); print('\nSaved files:')
for dp, _, fs in os.walk(OUT_DIR):
    for f in sorted(fs): print('  ', os.path.relpath(os.path.join(dp, f), WORK_DIR))

## 22 · (Optional) Chest X-ray branch — NIH ChestX-ray14 + Grad-CAM
NIH X-rays are *different patients* from PhysioNet, so they are **not** fused here. This trains/evaluates the image
encoder on its own task (60k images, 4 epochs, ~50 min on a T4); section 23 adds confidence intervals and Grad-CAM.
With MIMIC-IV + MIMIC-CXR it becomes the 4th modality.

In [ ]:
NIH_LABELS = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
              'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
IMG = dict(train_n=400 if QUICK else 60000, test_n=100 if QUICK else 8000, epochs=1 if QUICK else 4,
           size=128 if QUICK else 224, bs=16 if QUICK else 64)
entry = [p for p in find_files(INPUT_ROOT, ('.csv',)) if os.path.basename(p) == 'Data_Entry_2017.csv']
if not (RUN_IMAGING and entry):
    print('Imaging branch skipped (NIH chest X-ray dataset not attached or RUN_IMAGING=False).')
else:
    from PIL import Image
    from torchvision import models, transforms as T
    from sklearn.model_selection import GroupShuffleSplit
    t0 = time.time(); paths = {}
    for dp, _, fns in os.walk(os.path.dirname(entry[0]), followlinks=True):
        for f in fns:
            if f.endswith('.png'): paths[f] = os.path.join(dp, f)
    meta = pd.read_csv(entry[0]); meta = meta[meta['Image Index'].isin(paths)].reset_index(drop=True)
    Yimg = np.stack([meta['Finding Labels'].str.contains(l, regex=False).values for l in NIH_LABELS], 1).astype(np.float32)
    gi_tr, gi_te = next(GroupShuffleSplit(1, test_size=0.15, random_state=SEED).split(meta, groups=meta['Patient ID']))
    rng = np.random.default_rng(SEED)
    gi_tr = rng.choice(gi_tr, min(IMG['train_n'], len(gi_tr)), replace=False); gi_te = rng.choice(gi_te, min(IMG['test_n'], len(gi_te)), replace=False)
    print(f'{len(paths):,} X-rays found | using {len(gi_tr):,} train / {len(gi_te):,} test (patient-level split)')
    norm_ = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    tf_train = T.Compose([T.Resize((IMG['size'], IMG['size'])), T.RandomHorizontalFlip(), T.ToTensor(), norm_])
    tf_test  = T.Compose([T.Resize((IMG['size'], IMG['size'])), T.ToTensor(), norm_])
    class CXR(torch.utils.data.Dataset):
        def __init__(self, idx, tf): self.idx, self.tf = idx, tf
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            j = self.idx[i]; im = Image.open(paths[meta['Image Index'].iat[j]]).convert('RGB')
            return self.tf(im), torch.from_numpy(Yimg[j])
    nw = min(4, os.cpu_count() or 2)
    dl_tr = torch.utils.data.DataLoader(CXR(gi_tr, tf_train), batch_size=IMG['bs'], shuffle=True, num_workers=nw, pin_memory=True)
    dl_te = torch.utils.data.DataLoader(CXR(gi_te, tf_test), batch_size=IMG['bs'], shuffle=False, num_workers=nw, pin_memory=True)
    cnn = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if IMG_PRETRAINED else None)
    cnn.classifier = nn.Linear(1024, len(NIH_LABELS)); cnn = cnn.to(device)
    opt = torch.optim.AdamW(cnn.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
    for ep in range(IMG['epochs']):
        cnn.train(); tot = 0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device)
            with torch.autocast(device.type, enabled=device.type == 'cuda'):
                loss = F.binary_cross_entropy_with_logits(cnn(xb), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item() * len(xb)
        print(f'[cxr] epoch {ep+1} | loss {tot/len(gi_tr):.4f} | {time.time()-t0:.0f}s')
    cnn.eval(); P = []
    with torch.no_grad():
        for xb, _ in dl_te: P.append(torch.sigmoid(cnn(xb.to(device)).float()).cpu().numpy())
    P = np.concatenate(P); Yt = Yimg[gi_te]
    cxr_auc = {l: roc_auc_score(Yt[:, k], P[:, k]) for k, l in enumerate(NIH_LABELS) if 0 < Yt[:, k].sum() < len(Yt)}
    CXR_RES = pd.Series(cxr_auc, name='AUROC').sort_values(ascending=False); CXR_RES.to_csv(os.path.join(OUT_DIR, 'cxr_auroc_test.csv'))
    print(f'Mean AUROC over {len(cxr_auc)} findings: {CXR_RES.mean():.3f}'); print(CXR_RES.round(3).to_string())
    torch.save(cnn.state_dict(), os.path.join(OUT_DIR, 'cxr_densenet121.pt'))

## 23 · X-ray results with 95% CIs + Grad-CAM (Cardiomegaly, Effusion)
Per-finding AUROC with patient-level bootstrap CIs and number of test positives (rare findings like Hernia get wide CIs).
Grad-CAM is drawn for two large, common, visible findings on their most confident true-positive test images.

In [ ]:
# ============ CELL 16: per-finding AUROC with 95% CI + Grad-CAM on well-represented findings ============
# Reuses the DenseNet trained in cell 15 (from memory, or reloads results/cxr_densenet121.pt). No retraining.
import os, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image
from torchvision import models, transforms as T
warnings.filterwarnings('ignore')

INPUT_ROOT = os.environ.get('SEPSIS_INPUT_ROOT', '/kaggle/input')
WORK_DIR   = os.environ.get('SEPSIS_WORK_DIR', '/kaggle/working')
QUICK      = os.environ.get('SEPSIS_QUICK', '0') == '1'
OUT_DIR = os.path.join(WORK_DIR, 'results'); FIG_DIR = os.path.join(OUT_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SHOW = ['Cardiomegaly', 'Effusion']      # large, well-localised findings with many test positives

MODEL_PATH = os.path.join(OUT_DIR, 'cxr_densenet121.pt')
if not (all(n in globals() for n in ['cnn', 'P', 'Yt', 'gi_te', 'meta', 'paths', 'tf_test', 'IMG', 'NIH_LABELS']) or os.path.exists(MODEL_PATH)):
    print('No trained X-ray model in memory or on disk -> run the X-ray training cell first (same session). Skipping.')
else:
    needed = ['cnn', 'P', 'Yt', 'gi_te', 'meta', 'paths', 'tf_test', 'IMG', 'NIH_LABELS']
    if all(n in globals() for n in needed):
        print('Using the model and predictions still in memory from cell 15')
    else:
        print('Kernel was restarted: rebuilding the same test set and reloading the saved model (~3-5 min) ...')
        t0 = time.time()
        NIH_LABELS = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
                      'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
        IMG = dict(train_n=400 if QUICK else 60000, test_n=100 if QUICK else 8000, epochs=1 if QUICK else 4,
                   size=128 if QUICK else 224, bs=16 if QUICK else 64)                      # must match cell 15
        entry = None
        for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):
            dns[:] = [d for d in dns if not d.lower().startswith('images')]
            if 'Data_Entry_2017.csv' in fns: entry = os.path.join(dp, 'Data_Entry_2017.csv'); break
        assert entry, 'NIH chest X-ray dataset not attached'
        paths = {}
        for dp, _, fns in os.walk(os.path.dirname(entry), followlinks=True):
            for f in fns:
                if f.endswith('.png'): paths[f] = os.path.join(dp, f)
        meta = pd.read_csv(entry); meta = meta[meta['Image Index'].isin(paths)].reset_index(drop=True)
        Yimg = np.stack([meta['Finding Labels'].str.contains(l, regex=False).values for l in NIH_LABELS], 1).astype(np.float32)
        gi_tr, gi_te = next(GroupShuffleSplit(1, test_size=0.15, random_state=SEED).split(meta, groups=meta['Patient ID']))
        rng = np.random.default_rng(SEED)                                                    # same two draws as cell 15
        gi_tr = rng.choice(gi_tr, min(IMG['train_n'], len(gi_tr)), replace=False)
        gi_te = rng.choice(gi_te, min(IMG['test_n'], len(gi_te)), replace=False)
        tf_test = T.Compose([T.Resize((IMG['size'], IMG['size'])), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        cnn = models.densenet121(weights=None); cnn.classifier = nn.Linear(1024, len(NIH_LABELS))
        cnn.load_state_dict(torch.load(MODEL_PATH, map_location='cpu')); cnn = cnn.to(device).eval()
        class CXRTest(torch.utils.data.Dataset):
            def __len__(self): return len(gi_te)
            def __getitem__(self, i): return tf_test(Image.open(paths[meta['Image Index'].iat[gi_te[i]]]).convert('RGB'))
        dl = torch.utils.data.DataLoader(CXRTest(), batch_size=IMG['bs'], num_workers=min(4, os.cpu_count() or 2))
        P = []
        with torch.no_grad():
            for xb in dl: P.append(torch.sigmoid(cnn(xb.to(device)).float()).cpu().numpy())
        P = np.concatenate(P); Yt = Yimg[gi_te]
        print(f'rebuilt in {time.time()-t0:.0f}s')

    # ---- per-finding AUROC with patient-level bootstrap 95% CI ----
    pid = meta['Patient ID'].values[gi_te]
    groups = list(pd.Series(np.arange(len(pid))).groupby(pid).indices.values())
    rng_b = np.random.default_rng(0); B = 200
    boot = {l: [] for l in NIH_LABELS}
    for _ in range(B):
        idx = np.concatenate([groups[g] for g in rng_b.integers(0, len(groups), len(groups))])
        for k, l in enumerate(NIH_LABELS):
            y = Yt[idx, k]
            if 0 < y.sum() < len(y): boot[l].append(roc_auc_score(y, P[idx, k]))
    rows = []
    for k, l in enumerate(NIH_LABELS):
        y = Yt[:, k]; npos = int(y.sum())
        if not 0 < npos < len(y): continue
        lo, hi = np.percentile(boot[l], [2.5, 97.5])
        rows.append({'finding': l, 'test positives': npos, 'prevalence %': round(float(100 * y.mean()), 2),
                     'AUROC': round(roc_auc_score(y, P[:, k]), 3), '95% CI': f'{lo:.3f}–{hi:.3f}', 'CI width': round(hi - lo, 3)})
    CXR_CI = pd.DataFrame(rows).sort_values('AUROC', ascending=False).set_index('finding')
    CXR_CI.to_csv(os.path.join(OUT_DIR, 'cxr_auroc_ci_test.csv'))
    print(f'{len(gi_te):,} test images from {len(groups):,} patients | mean AUROC {CXR_CI.AUROC.mean():.3f}')
    print(CXR_CI.to_string())

    # ---- Grad-CAM on the 3 most confident TRUE-positive test images per finding ----
    store = {}
    def fhook(mod, inp, out):
        store['act'] = out; out.register_hook(lambda g: store.__setitem__('grad', g))
    hk = cnn.features.denseblock4.register_forward_hook(fhook)
    cnn.eval()
    targets = [t for t in SHOW if Yt[:, NIH_LABELS.index(t)].sum() > 0]
    fig, axs = plt.subplots(len(targets), 3, figsize=(12, 4.6 * len(targets)), squeeze=False, gridspec_kw={'hspace': 0.3, 'wspace': 0.05})
    for r, target in enumerate(targets):
        k = NIH_LABELS.index(target); prev = Yt[:, k].mean()
        pos = np.flatnonzero(Yt[:, k] == 1); pick = pos[np.argsort(-P[pos, k])[:3]]
        for c in range(3):
            a = axs[r, c]; a.axis('off')
            if c >= len(pick): continue
            j = gi_te[pick[c]]; im = Image.open(paths[meta['Image Index'].iat[j]]).convert('RGB')
            xb = tf_test(im)[None].to(device); cnn.zero_grad()
            out = cnn(xb); out[0, k].backward()
            w = store['grad'].mean((2, 3), keepdim=True); cam = F.relu((w * store['act']).sum(1, keepdim=True))
            cam = F.interpolate(cam, size=(IMG['size'], IMG['size']), mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            p = torch.sigmoid(out[0, k]).item()
            a.imshow(im.convert('L').resize((IMG['size'], IMG['size'])), cmap='gray'); a.imshow(cam, cmap='jet', alpha=0.4)
            a.set_title(f"{target}: p={p:.2f} ({p/prev:.0f}× prevalence)\nlabel: {meta['Finding Labels'].iat[j]}", fontsize=8)
    hk.remove()
    plt.suptitle('Grad-CAM, image branch (NIH ChestX-ray14, held-out patients)', y=0.95)
    plt.savefig(os.path.join(FIG_DIR, '08_gradcam.png'), dpi=160, bbox_inches='tight'); plt.show()